# CMP4509 — MNIST üzerinde PGD ile dayanıklı öğrenme

Bu notebook ile bütün zinciri yeniden çalıştırabiliriz: **veri → CNN → normal eğitim → saldırı → PGD eğitimi → daha güçlü test → CSV/grafik**.

Bu notebook **kendi kaynak kodunu içerir**. Colab/Kaggle'da başka bir depo klonlamadan, yukarıdan aşağı çalışır. Varsayılan `smoke`, küçük bir çalışma kontrolüdür; bilimsel sonuç değildir. Dokuz tarihî modelin gerçek final-test kayıtları depodaki `results/FINAL_TEST_ALL_MODELS.csv` dosyasında bulunur. Bu notebook yeni koşular üretir; eski checkpoint'leri veya sonuçları yeniden üretmiş sayılmaz.

Arşivlenmiş dokuz kaydın istatistiğini eğitim yapmadan kontrol etmek için depodaki `02_verify_recorded_results.ipynb` kullanılır; bu aritmetik kontrol, tarihî saldırı çalışmasını yeniden yürütmez.

Bu notebook'un kaynak dışa aktarımı eğitim için gereken dosyaları içerir. Gömülü README'deki rapor, diğer notebook ve denetim belgesi bağlantıları [tam GitHub deposuna](https://github.com/emirbakac/cmp4509-madry-mnist) aittir; bu ek teslim dosyaları yerel kaynak dışa aktarımına dahil değildir.

Kaynaklar: [Madry ve ark., makale](https://arxiv.org/abs/1706.06083), [yazarların MNIST uygulaması](https://github.com/MadryLab/mnist_challenge).

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Settings → Accelerator → GPU, ilk MNIST indirmesi için Internet açık. GPU kotası ve erişim garanti değildir. `smoke` CPU'da çalışabilir; smoke dışındaki config'ler için bu notebook CUDA GPU ister.

Rapor: **13 Aralık 2026, 23:59**; gerçek `CMP4509_report_template.zip` şablonu kullanılacak. Sunum: **30 Kasım veya 7 Aralık**, QA dahil **10 dakika**. Public GitHub bağlantısı raporda bulunmalı.


## 1 · Ortamı kontrol et

Önceden kurulmuş GPU destekli PyTorch'u koruyoruz. Yalnız eksik paketler kurulur. Birbiriyle uyumsuz torch/torchvision varsa hücre açık hata verir; temiz bir GPU oturumunda yeniden başlatmak genellikle en kolay başlangıçtır.


In [ ]:
import importlib.util
import subprocess
import sys

missing_torch = importlib.util.find_spec("torch") is None
missing_vision = importlib.util.find_spec("torchvision") is None
if missing_torch:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch>=2.5,<3", "torchvision>=0.20,<1"])
elif missing_vision:
    raise RuntimeError(
        "PyTorch var ama torchvision eksik. Torch sürümünüzle uyumlu torchvision kurun "
        "veya temiz Colab/Kaggle GPU oturumu açın; GPU paketini otomatik değiştirmiyoruz."
    )
for package, requirement in {"numpy": "numpy>=1.26,<3", "matplotlib": "matplotlib>=3.8,<4", "pandas": "pandas>=2.2,<4"}.items():
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement])

try:
    import torch
    import torchvision
    import numpy as np
    import matplotlib
    import pandas as pd
except Exception as exc:
    raise RuntimeError(
        "Paketler uyumsuz veya eksik. Temiz bir Colab/Kaggle GPU oturumunda yeniden deneyin. "
        "Yerelde PyTorch'un resmî kurulumundan birbiriyle uyumlu torch/torchvision seçin."
    ) from exc

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2 · Deney ayarları ve kalıcı kayıt

İlk çalıştırmada `smoke` ile ilerle. `main`, baseline ve PGD40 için üç seed; `main_curriculum_pgd10_to_pgd40`, entegre PGD10→PGD40 için üç seed kullanır. Bu config'ler **ayrı notebook koşularıdır**. `main_pgd10_probe` isteğe bağlı üç epoch araştırma koşusudur; entegre curriculum için önce çalıştırılması gerekmez. Tarihî ölçümler ayrı probe checkpoint'lerinden transferle elde edilmiştir; entegre programın o dosyalarla bit düzeyinde aynı sonucu vermesi doğrulanmamıştır. **Eğitim ve model seçimi doğrulama setiyle yapılır.** `EPOCHS_PER_CALL=1` ile ücretsiz GPU oturumlarında eğitimi birer epoch parçalara ayırabilirsin; test bütün epoch'lar tamamlanana kadar bekler. `pilot/main` testini ancak bütün ayarları dondurunca `PROTOCOL_FROZEN = True` ile aç.

Final karşılaştırmada `CONFIG_NAME = "main"` ve ardından `"main_curriculum_pgd10_to_pgd40"` koşularını kullan; her birinin testinde `PROTOCOL_FROZEN = True`, `FINAL_STRONG_EVAL = True` seç: 100 PGD adımı × 5 restart. Önceden eğitilmiş checkpoint varsa eğitim son tamamlanan epoch'tan devam eder; tamamlanmışsa yalnız değerlendirme ilerler.

Colab'da uzun koşudan önce `USE_GOOGLE_DRIVE = True` yap. Drive bağlamak kullanıcı girişini gerektirebilir. Kaggle çıktıları `/kaggle/working` altında saklanır; oturum sonunda **Save Version** yap. Aynı kayıt kökünde config'ini değiştirme: değiştirilmiş deney için yeni config adı/kayıt kökü kullan.


In [ ]:
from pathlib import Path
import os

CONFIG_NAME = "smoke"                    # smoke, pilot, main, main_pgd10_probe, main_curriculum_pgd10_to_pgd40
MODES = (
    ["baseline", "adversarial"]
    if CONFIG_NAME in {"smoke", "pilot", "main"}
    else ["adversarial"]
)
SEEDS = [0, 1, 2] if CONFIG_NAME in {
    "main",
    "main_pgd10_probe",
    "main_curriculum_pgd10_to_pgd40"
} else [0]
PROTOCOL_FROZEN = False                   # Pilot/main testi: kararlar bittikten sonra True
FINAL_STRONG_EVAL = False                 # Final: PGD100 x 5 restart
USE_GOOGLE_DRIVE = False                  # Colab uzun koşularında True önerilir
RESUME = True                            # last.pt varsa oradan devam et
EPOCHS_PER_CALL = 0                       # 0=hepsi; örn. 1 ile oturum başına bir epoch
SYNTHETIC = False                         # Yalnız çevrimdışı smoke; MNIST sonucu değildir

assert CONFIG_NAME in {
    "smoke",
    "pilot",
    "main",
    "main_pgd10_probe",
    "main_curriculum_pgd10_to_pgd40"
}
assert MODES and set(MODES) <= {"baseline", "adversarial"}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
if CONFIG_NAME != "smoke" and not torch.cuda.is_available():
    raise RuntimeError("pilot/main için CUDA GPU açın veya CONFIG_NAME='smoke' ile başlayın.")
if SYNTHETIC and CONFIG_NAME != "smoke":
    raise ValueError("Sentetik veri yalnız smoke kontrolü içindir.")
if FINAL_STRONG_EVAL and not PROTOCOL_FROZEN:
    raise ValueError("Final test için önce ayarları dondurun ve PROTOCOL_FROZEN=True yapın.")

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
IS_KAGGLE = Path("/kaggle/working").exists()
if "_NOTEBOOK_BASE_DIR" not in globals():
    _NOTEBOOK_BASE_DIR = Path.cwd()
BASE_DIR = Path("/content") if IS_COLAB else (Path("/kaggle/working") if IS_KAGGLE else _NOTEBOOK_BASE_DIR)
PROJECT_DIR = BASE_DIR / "madry_mnist_project"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
if USE_GOOGLE_DRIVE:
    if not IS_COLAB:
        raise RuntimeError("USE_GOOGLE_DRIVE yalnız Colab içindir.")
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/CMP4509_madry")
else:
    ARTIFACT_ROOT = BASE_DIR / "madry_mnist_outputs"
RUN_CONFIG_NAME = CONFIG_NAME + ("_synthetic" if SYNTHETIC else "")
RUN_ROOT = ARTIFACT_ROOT / ("synthetic_runs" if SYNTHETIC else "runs")
RESULTS_DIR = ARTIFACT_ROOT / ("synthetic_results" if SYNTHETIC else "results") / RUN_CONFIG_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
for name in ["mnist_robust", "configs", "docs", "results"]:
    Path(name).mkdir(exist_ok=True)

RUN_TEST_EVALUATION = CONFIG_NAME == "smoke" or PROTOCOL_FROZEN
if RUN_TEST_EVALUATION and CONFIG_NAME in {"main", "main_curriculum_pgd10_to_pgd40"} and not FINAL_STRONG_EVAL:
    raise ValueError("Final main/curriculum testi için FINAL_STRONG_EVAL=True seçin: PGD100 x 5. Config varsayılanı 10 restart tarihî uyumluluk için korunur.")
print("Kaynak:", PROJECT_DIR)
print("Kalıcı/indirilecek çıktılar:", ARTIFACT_ROOT)
print("Config:", CONFIG_NAME, "| Modeller:", MODES, "| Seed'ler:", SEEDS)
print("Test açık mı?", RUN_TEST_EVALUATION, "| Final güçlü test?", FINAL_STRONG_EVAL)
print("Ekran görüntüsü zamanı: bu ayarlar ve GPU bilgisi deney kaydınız olsun.")


## 3 · Kaynak kodunu oluştur

Aşağıdaki hücreler projedeki kaynakları görünür biçimde yazar. `Run all` için değiştirmeye gerek yok. Öğrenirken özellikle model, saldırı ve eğitim dosyalarını açıp birlikte okuyabiliriz.

Makalenin hedefi:

\[
\min_\theta \; \mathbb{E}_{(x,y)} [ \max_{\|\delta\|_\infty\leq 0.3} \mathcal{L}(f_\theta(x+\delta),y)] .
\]

İçte PGD, mevcut modeli zorlayacak bir görüntü arar. Dışta optimizer, modelin bu zor görüntüdeki kaybını düşürür. Piksel ölçeği `[0,1]` olduğu için ε=0.3 bu ölçekte uygulanır. Her PGD adımından sonra hem ε sınırına hem `[0,1]` aralığına geri dönülür.

Resmî kod: 32/64 convolution, 1024 gizli birim, Adam `1e-4`, batch 50, 100.000 güncelleme; eğitim PGD40, α=.01, ε=.3. Bizim `main` başlangıcımız daha küçük ağ/10 epoch/Adam `1e-3`/batch128 kullanır. **Bu farklar raporlanır; birebir sayısal tekrar iddiası kurulmaz.**


In [ ]:
%%writefile mnist_robust/__init__.py
"""MNIST üzerinde küçük ve tekrar çalıştırılabilir adversarial training projesi."""

from .attacks import fgsm, pgd_linf
from .model import SmallCNN

__all__ = ["SmallCNN", "fgsm", "pgd_linf"]
__version__ = "0.1.0"


In [ ]:
%%writefile mnist_robust/attacks.py
"""Untargeted FGSM ve L-infinity PGD; tehdit modeli ham [0, 1] piksellerde.

Değerlendirmede her örnek için bulunan herhangi bir yanlış sınıflandırmayı
koruruz. Böylece son iterasyonun tesadüfen tekrar doğru sınıflandırılması,
ölçülen robust accuracy değerini yapay olarak yükseltmez.
"""

from contextlib import contextmanager
import math
from numbers import Real

import torch
from torch import nn
from torch.nn import functional as F


@contextmanager
def _attack_mode(model: nn.Module):
    # Mixed train/eval alt modül ayarlarını da geri yükle. BN istatistikleri
    # saldırı üretilirken güncellenmez; dış eğitim adımı bağımsız kalır.
    modes = [(module, module.training) for module in model.modules()]
    model.eval()
    try:
        yield
    finally:
        for module, training in modes:
            module.training = training


def _nonnegative_finite(name, value):
    if isinstance(value, bool) or not isinstance(value, Real):
        raise ValueError(f"{name} sonlu ve negatif olmayan bir sayı olmalı.")
    if not math.isfinite(value) or value < 0:
        raise ValueError(f"{name} sonlu ve negatif olmayan bir sayı olmalı.")


def _validate_inputs(x, y, eps):
    _nonnegative_finite("eps", eps)
    if not isinstance(x, torch.Tensor) or not x.is_floating_point():
        raise ValueError("x kayan noktalı bir Tensor olmalı.")
    if x.ndim < 2 or x.shape[0] == 0:
        raise ValueError("x boş olmayan bir batch içermeli.")
    if not isinstance(y, torch.Tensor) or y.ndim != 1 or len(y) != len(x):
        raise ValueError("y her giriş için bir sınıf etiketi içermeli.")
    if y.dtype != torch.long or y.device != x.device:
        raise ValueError("y torch.long olmalı ve x ile aynı cihazda bulunmalı.")
    if not torch.isfinite(x).all() or torch.any(x < 0) or torch.any(x > 1):
        raise ValueError("Saldırı ham, sonlu [0, 1] pikseller bekliyor.")


def _project(candidate, clean, eps):
    return torch.maximum(torch.minimum(candidate, clean + eps), clean - eps).clamp(0, 1)


def _select_candidate(best, best_loss, best_wrong, candidate, logits, losses, y):
    """Önce başarılı saldırı, sonra cross-entropy: örnek bazında seçim."""
    with torch.no_grad():
        wrong = logits.argmax(dim=1).ne(y)
        replace = (wrong & ~best_wrong) | ((wrong == best_wrong) & (losses > best_loss))
        broadcast = replace.reshape((-1,) + (1,) * (candidate.ndim - 1))
        best = torch.where(broadcast, candidate.detach(), best)
        best_loss = torch.where(replace, losses.detach(), best_loss)
        best_wrong = torch.where(replace, wrong, best_wrong)
    return best, best_loss, best_wrong


def fgsm(model: nn.Module, x: torch.Tensor, y: torch.Tensor, eps: float):
    """Bir gradient-sign adımı; modelin modu ve parametre .grad'ları korunur."""
    _validate_inputs(x, y, eps)
    clean = x.detach()
    if eps == 0:
        return clean.clone()
    with _attack_mode(model), torch.enable_grad():
        candidate = clean.clone().requires_grad_(True)
        loss = F.cross_entropy(model(candidate), y, reduction="sum")
        gradient, = torch.autograd.grad(loss, candidate, only_inputs=True)
        return _project(candidate.detach() + eps * gradient.sign(), clean, eps).detach()


def pgd_linf(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 0.3,
    alpha: float = 0.01,
    steps: int = 40,
    restarts: int = 1,
    random_start: bool = True,
    track_best: bool = True,
):
    """Untargeted L-infinity PGD.

    track_best=True: temiz girişi, her iterasyonu ve restart'ı karşılaştırır;
    bulunan yanlış sınıflandırmayı her örnek için korur. Değerlendirme içindir.
    track_best=False: tek restart'ın son iterasyonunu döndürür. Adversarial
    training için kullanılır ve restarts=1 gerektirir.

    Dönen Tensor graph'tan ayrıdır. Giriş ya da model parametrelerinin .grad
    alanlarına yazılmaz. torch.no_grad() içinden de çağrılabilir; inference_mode
    içinde çağırmayın, çünkü saldırı giriş türevi gerektirir.
    """
    _validate_inputs(x, y, eps)
    _nonnegative_finite("alpha", alpha)
    for name, value, minimum in (("steps", steps, 0), ("restarts", restarts, 1)):
        if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
            raise ValueError(f"{name} en az {minimum} olan bir tamsayı olmalı.")
    if not isinstance(random_start, bool) or not isinstance(track_best, bool):
        raise ValueError("random_start ve track_best bool olmalı.")
    if not track_best and restarts != 1:
        raise ValueError("track_best=False son iterasyon içindir; restarts=1 kullanın.")
    clean = x.detach()
    if eps == 0:
        return clean.clone()

    with _attack_mode(model):
        if track_best:
            with torch.no_grad():
                logits = model(clean)
                best = clean.clone()
                best_loss = F.cross_entropy(logits, y, reduction="none")
                best_wrong = logits.argmax(dim=1).ne(y)

        for _ in range(restarts):
            if random_start:
                candidate = _project(clean + torch.empty_like(clean).uniform_(-eps, eps), clean, eps)
            else:
                candidate = clean.clone()

            for _ in range(steps):
                with torch.enable_grad():
                    candidate = candidate.detach().requires_grad_(True)
                    logits = model(candidate)
                    losses = F.cross_entropy(logits, y, reduction="none")
                    if track_best:
                        best, best_loss, best_wrong = _select_candidate(
                            best, best_loss, best_wrong, candidate, logits, losses, y
                        )
                    gradient, = torch.autograd.grad(losses.sum(), candidate, only_inputs=True)
                candidate = _project(candidate.detach() + alpha * gradient.sign(), clean, eps).detach()

            if track_best:
                with torch.no_grad():
                    logits = model(candidate)
                    losses = F.cross_entropy(logits, y, reduction="none")
                    best, best_loss, best_wrong = _select_candidate(
                        best, best_loss, best_wrong, candidate, logits, losses, y
                    )

        return (best if track_best else candidate).detach()


In [ ]:
%%writefile mnist_robust/data.py
"""Sabit MNIST train/validation ayrımı; test seti model seçimi için kullanılmaz."""

from pathlib import Path
import random

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset, random_split


def _seed_worker(_worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)


def _positive_int(name, value, allow_none=False):
    if allow_none and value is None:
        return
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} pozitif bir tamsayı olmalı.")


def _limit(dataset, count, name):
    if count is None:
        return dataset
    if count > len(dataset):
        raise ValueError(f"{name}={count}, mevcut örnek sayısını ({len(dataset)}) aşıyor.")
    return Subset(dataset, range(count))


def _synthetic_split(size, seed):
    # Sadece çevrimdışı yazılım smoke testi. MNIST verisi ya da bilimsel sonuç değil.
    # Ayrı RNG'ler, limit küçültüldüğünde örnek/etiket eşleşmesini sabit tutar.
    images = torch.rand(size, 1, 28, 28, generator=torch.Generator().manual_seed(seed))
    labels = torch.randint(10, (size,), generator=torch.Generator().manual_seed(seed + 1000))
    return TensorDataset(images, labels)


def make_loaders(
    data_dir,
    batch_size=128,
    seed=0,
    split_seed=2026,
    val_size=5000,
    train_limit=None,
    val_limit=None,
    test_limit=None,
    num_workers=0,
    synthetic=False,
):
    """{'train', 'val', 'test'} DataLoader sözlüğü döndürür.

    split_seed veriyi ayırır; seed yalnızca batch sırasını/worker RNG'lerini
    yönetir. Dolayısıyla farklı eğitim seed'leri aynı validation verisini görür.
    Piksel ölçeği [0, 1]'dir: epsilon=0.3 doğrudan bu ölçekte uygulanır.
    synthetic=True rastgele görüntü/etiket üretir ve yalnızca smoke test içindir.
    """
    _positive_int("batch_size", batch_size)
    _positive_int("val_size", val_size)
    for name, limit in (("train_limit", train_limit), ("val_limit", val_limit), ("test_limit", test_limit)):
        _positive_int(name, limit, allow_none=True)
    if isinstance(num_workers, bool) or not isinstance(num_workers, int) or num_workers < 0:
        raise ValueError("num_workers negatif olmayan bir tamsayı olmalı.")
    for name, value in (("seed", seed), ("split_seed", split_seed)):
        if isinstance(value, bool) or not isinstance(value, int) or not 0 <= value < 2**32:
            raise ValueError(f"{name}, [0, 2**32) aralığında bir tamsayı olmalı.")

    if synthetic:
        datasets = {
            "train": _synthetic_split(train_limit or 256, split_seed),
            "val": _synthetic_split(val_limit or 64, split_seed + 1),
            "test": _synthetic_split(test_limit or 64, split_seed + 2),
        }
    else:
        # Ortamda torchvision bozuk olsa bile offline attack testleri çalışabilir.
        from torchvision.datasets import MNIST
        from torchvision.transforms import ToTensor

        data_dir = Path(data_dir).expanduser()
        full_train = MNIST(str(data_dir), train=True, download=True, transform=ToTensor())
        if val_size >= len(full_train):
            raise ValueError("val_size, toplam eğitim örnek sayısından küçük olmalı.")
        train, val = random_split(
            full_train,
            [len(full_train) - val_size, val_size],
            generator=torch.Generator().manual_seed(split_seed),
        )
        test = MNIST(str(data_dir), train=False, download=True, transform=ToTensor())
        datasets = {
            "train": _limit(train, train_limit, "train_limit"),
            "val": _limit(val, val_limit, "val_limit"),
            "test": _limit(test, test_limit, "test_limit"),
        }

    return {
        name: DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=name == "train",
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            worker_init_fn=_seed_worker,
            generator=torch.Generator().manual_seed(seed + index),
            persistent_workers=False,
        )
        for index, (name, dataset) in enumerate(datasets.items())
    }


In [ ]:
%%writefile mnist_robust/experiment.py
"""Train on train/validation only; evaluate frozen checkpoints on the test split."""
import argparse
import csv
import hashlib
import json
import os
import random
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from .attacks import fgsm, pgd_linf
from .data import make_loaders
from .model import SmallCNN
from .utils import environment_info, resolve_device, seed_everything, source_fingerprint


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n')


def write_csv(path, rows):
    if not rows:
        return
    path = Path(path)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    try:
        with temporary.open('w', newline='') as f:
            # Resuming historical checkpoints may add newly recorded columns.
            fields = list(dict.fromkeys(key for row in rows for key in row))
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def rng_state(loader=None):
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
            'loader': loader.generator.get_state() if loader is not None else None}


def restore_rng(state, loader=None):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])
    if state.get('mps') is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(state['mps'].cpu())
    if loader is not None and state['loader'] is not None:
        loader.generator.set_state(state['loader'].cpu())


def save_checkpoint(path, state):
    temporary = path.with_suffix('.tmp')
    torch.save(state, temporary)
    os.replace(temporary, path)


def training_steps(cfg, epoch):
    """epoch is zero-based: curriculum uses PGD10 for 0,1,2, then PGD40."""
    if not isinstance(epoch, int) or isinstance(epoch, bool) or epoch < 0:
        raise ValueError('epoch must be a nonnegative integer.')
    curriculum = cfg.get('curriculum')
    if curriculum is None:
        return cfg['train_steps']
    stage = 'epochs_1_to_3' if epoch < 3 else 'epochs_4_to_10'
    return curriculum[stage]['train_steps']


def validate(model, loader, device, cfg, seed):
    """Fixed validation attack RNG; validation does not consume training RNG."""
    previous = rng_state()
    seed_everything(seed)
    was_training = model.training
    model.eval()
    total = clean_correct = robust_correct = 0
    try:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                clean = model(x).argmax(1).eq(y)
            adv = pgd_linf(model, x, y, eps=cfg['epsilon'], alpha=cfg['eval_alpha'],
                           steps=cfg['val_steps'], restarts=cfg['val_restarts'])
            with torch.no_grad():
                robust = clean & model(adv).argmax(1).eq(y)
            total += len(y)
            clean_correct += clean.sum().item()
            robust_correct += robust.sum().item()
    finally:
        model.train(was_training)
        restore_rng(previous)
    return clean_correct / total, robust_correct / total


def train(model, loaders, device, cfg, args, run_dir):
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    start, best_score, history = 0, -1.0, []
    last = run_dir / 'last.pt'
    if args.resume and not last.is_file():
        raise FileNotFoundError(f'Cannot resume: {last} is missing. Check the run directory before starting a new run.')
    if not last.exists() and any((run_dir / name).exists() for name in ('best.pt', 'history.csv')):
        raise FileExistsError('Run contains best.pt/history.csv but no last.pt. Preserve these partial or archived artifacts; use --eval-only for best.pt or a new --output-dir.')
    if last.exists():
        if not args.resume:
            raise FileExistsError(f'{last} exists. Use --resume or a new --output-dir.')
        state = torch.load(last, map_location='cpu', weights_only=False)
        if state['config'] != cfg or state['mode'] != args.mode or state['seed'] != args.seed or state['synthetic'] != args.synthetic:
            raise ValueError('Resume config/mode/seed/dataset differs from saved checkpoint.')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        for values in optimizer.state.values():
            for key, value in values.items():
                if torch.is_tensor(value):
                    values[key] = value.to(device)
        start, best_score, history = state['epoch'], state['best_score'], state['history']
        restore_rng(state['rng'], loaders['train'])
        # A process can stop after publishing last.pt but before its CSV. The
        # checkpoint is authoritative, including when all epochs are complete.
        write_csv(run_dir / 'history.csv', history)
    end = cfg['epochs']
    if args.epochs_per_call:
        end = min(end, start + args.epochs_per_call)
    for epoch in range(start, end):
        begin = time.monotonic()
        model.train()
        steps = training_steps(cfg, epoch) if args.mode == 'adversarial' else 0
        loss_sum, correct, total = 0., 0, 0
        for x, y in loaders['train']:
            x, y = x.to(device), y.to(device)
            if args.mode == 'adversarial':
                x = pgd_linf(
                    model, x, y,
                    eps=cfg['epsilon'],
                    alpha=cfg['train_alpha'],
                    steps=steps,
                    restarts=1,
                    track_best=False
                )
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += logits.argmax(1).eq(y).sum().item()
            total += len(y)
        val_clean, val_pgd = validate(model, loaders['val'], device, cfg, cfg['eval_seed'])
        score = val_clean if args.mode == 'baseline' else val_pgd
        improved = score > best_score
        best_score = max(best_score, score)
        row = {'epoch': epoch + 1, 'train_loss': loss_sum / total,
               'train_pgd_steps': steps,
               'train_accuracy': correct / total, 'val_clean_accuracy': val_clean,
               'val_pgd_accuracy': val_pgd, 'seconds': time.monotonic() - begin}
        history.append(row)
        state = {'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                 'epoch': epoch + 1, 'best_score': best_score, 'history': history,
                 'config': cfg, 'seed': args.seed, 'mode': args.mode,
                 'synthetic': args.synthetic, 'rng': rng_state(loaders['train']),
                 'source': source_fingerprint()}
        # Publish best before advancing last: an interruption must not leave
        # last claiming an improvement whose best checkpoint was never saved.
        if improved:
            save_checkpoint(run_dir / 'best.pt', state)
        save_checkpoint(last, state)
        write_csv(run_dir / 'history.csv', history)
        print(json.dumps(row), flush=True)
    return end >= cfg['epochs']


def plot_examples(path, examples, epsilon):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    x, xf, xp, y, pc, pf, pp = examples
    n = min(len(x), 8)
    fig, axes = plt.subplots(4, n, figsize=(1.7 * n, 7), squeeze=False)
    for i in range(n):
        for row, (images, predictions, label) in enumerate([(x, pc, 'Clean'), (xf, pf, 'FGSM'), (xp, pp, 'PGD')]):
            axes[row, i].imshow(images[i, 0], cmap='gray', vmin=0, vmax=1)
            axes[row, i].set_title(f'{label}: {predictions[i].item()} / y={y[i].item()}', fontsize=8)
        axes[3, i].imshow((xp[i, 0] - x[i, 0]).abs(), cmap='magma', vmin=0, vmax=epsilon)
        axes[3, i].set_title('|PGD − clean|', fontsize=8)
        for row in range(4):
            axes[row, i].axis('off')
    fig.suptitle('First test examples (not cherry-picked); perturbation color range [0, ε]')
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def evaluate(model, loader, device, cfg, args, run_dir, checkpoint):
    model.eval()
    seed_everything(cfg['eval_seed'])
    steps = args.eval_steps if args.eval_steps is not None else cfg['eval_steps']
    restarts = args.eval_restarts if args.eval_restarts is not None else cfg['eval_restarts']
    max_training_steps = max(training_steps(cfg, epoch) for epoch in range(cfg['epochs']))
    if steps <= max_training_steps or restarts <= 1:
        raise ValueError('Final PGD evaluation requires more steps than training and >1 restart.')
    protocol = {'epsilon': cfg['epsilon'], 'alpha': cfg['eval_alpha'], 'steps': steps,
                'restarts': restarts, 'random_start': True, 'loss': 'cross_entropy',
                'candidate_selection': 'success_first_all_iterates_including_clean',
                'eval_seed': cfg['eval_seed'], 'batch_size': cfg['batch_size']}
    source = source_fingerprint()
    digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    label = f'pgd{steps}_r{restarts}'
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    eval_dir = run_dir / 'evaluations' / f'{label}_{stamp}_{uuid.uuid4().hex[:8]}'
    eval_dir.mkdir(parents=True, exist_ok=False)
    print(f'New evaluation output: {eval_dir}', flush=True)
    rows, examples = [], None
    begin = time.monotonic()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            pc = model(x).argmax(1)
        xf = fgsm(model, x, y, cfg['epsilon'])
        xp = pgd_linf(model, x, y, eps=cfg['epsilon'], alpha=cfg['eval_alpha'], steps=steps, restarts=restarts)
        with torch.no_grad():
            pf, pp = model(xf).argmax(1), model(xp).argmax(1)
        for j in range(len(y)):
            clean = int(pc[j].eq(y[j]).item())
            f = clean * int(pf[j].eq(y[j]).item())
            p = clean * int(pp[j].eq(y[j]).item())
            rows.append({'index': len(rows), 'label': y[j].item(), 'clean_correct': clean,
                         'fgsm_correct': f, 'pgd_correct': p, 'combined_correct': f * p})
        if examples is None:
            examples = tuple(t[:8].detach().cpu() for t in (x, xf, xp, y, pc, pf, pp))
        print(f'Evaluation {label}: {len(rows)}/{len(loader.dataset)}', flush=True)
    group = hashlib.sha256(json.dumps({'config': cfg, 'protocol': protocol, 'synthetic': args.synthetic,
                                      'source_sha256': source['sha256']}, sort_keys=True).encode()).hexdigest()[:16]
    result = {'config_name': cfg['name'], 'protocol_id': group, 'mode': args.mode,
              'seed': args.seed, 'dataset': 'synthetic' if args.synthetic else 'MNIST',
              'n_test': len(rows), 'clean_accuracy': sum(r['clean_correct'] for r in rows) / len(rows),
              'fgsm_accuracy': sum(r['fgsm_correct'] for r in rows) / len(rows),
              'pgd_accuracy': sum(r['pgd_correct'] for r in rows) / len(rows),
              'combined_accuracy': sum(r['combined_correct'] for r in rows) / len(rows),
              'eval_steps': steps, 'eval_restarts': restarts, 'epsilon': cfg['epsilon'],
              'checkpoint_sha256': digest, 'source_sha256': source['sha256'],
              'eval_seconds': time.monotonic() - begin}
    result['selected_epoch'] = state['epoch']
    detail = {'metrics': result, 'protocol': protocol, 'config': cfg,
              'checkpoint': str(checkpoint), 'source': source,
              'checkpoint_source': state.get('source'), 'environment': environment_info(),
              'device_selected': str(device), 'created_utc': datetime.now(timezone.utc).isoformat(),
              'interpretation': 'Smoke/synthetic is plumbing only; empirical attack accuracy is not certified robustness.'}
    write_json(eval_dir / 'evaluation.json', detail)
    write_csv(eval_dir / 'per_example.csv', rows)
    plot_examples(eval_dir / 'examples.png', examples, cfg['epsilon'])
    # metrics.csv marks a complete evaluation, so interrupted plots are not aggregated.
    write_csv(eval_dir / 'metrics.csv', [result])
    relative = eval_dir.relative_to(run_dir)
    pointer = {'evaluation_dir': str(relative), 'metrics': str(relative / 'metrics.csv'),
               'examples': str(relative / 'examples.png')}
    pointer_tmp = run_dir / f'.latest_evaluation_{uuid.uuid4().hex}.tmp'
    write_json(pointer_tmp, pointer)
    os.replace(pointer_tmp, run_dir / 'latest_evaluation.json')
    print(json.dumps(result, indent=2), flush=True)
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', required=True)
    parser.add_argument('--mode', choices=['baseline', 'adversarial'], default='baseline')
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--output-dir', default='runs')
    parser.add_argument('--data-dir', default='data')
    parser.add_argument('--device', default='auto', choices=['auto', 'cpu', 'cuda', 'mps'])
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--synthetic', action='store_true', help='Offline plumbing test only; never a scientific result.')
    group = parser.add_mutually_exclusive_group()
    group.add_argument('--train-only', action='store_true')
    group.add_argument('--eval-only', action='store_true')
    parser.add_argument('--checkpoint')
    parser.add_argument('--eval-steps', type=int)
    parser.add_argument('--eval-restarts', type=int)
    parser.add_argument('--epochs-per-call', type=int, default=0, help='0=all remaining; checkpointed chunks for free sessions.')
    args = parser.parse_args()
    cfg = json.loads(Path(args.config).read_text())
    if args.epochs_per_call < 0:
        parser.error('--epochs-per-call must be nonnegative')
    if args.synthetic and cfg['name'] != 'smoke':
        parser.error('--synthetic is restricted to the smoke configuration')
    if args.eval_only and not args.checkpoint:
        parser.error('--eval-only requires --checkpoint')
    if args.checkpoint and not args.eval_only:
        parser.error('--checkpoint is only for --eval-only; use --resume to continue training')
    seed_everything(args.seed)
    device = resolve_device(args.device)
    if device.type == 'cpu':
        torch.set_num_threads(min(torch.get_num_threads(), 4))
    run_dir = Path(args.output_dir) / (cfg['name'] + ('_synthetic' if args.synthetic else '')) / args.mode / f'seed_{args.seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f'Device={device}; run={run_dir}; synthetic={args.synthetic}', flush=True)
    model = SmallCNN(width=cfg['width'], hidden=cfg['hidden']).to(device)
    loaders = make_loaders(args.data_dir, batch_size=cfg['batch_size'], seed=args.seed,
                           split_seed=cfg['split_seed'], val_size=cfg['val_size'],
                           train_limit=cfg['train_limit'], val_limit=cfg['val_limit'], test_limit=cfg['test_limit'],
                           num_workers=cfg['num_workers'], synthetic=args.synthetic)
    if not args.eval_only:
        # Validate an existing run before changing its provenance files.
        config_path = run_dir / 'config.json'
        if config_path.exists() and json.loads(config_path.read_text()) != cfg:
            raise ValueError('Existing run has a different config; choose a new output directory.')
        write_json(config_path, cfg)
        env = environment_info()
        env.update({'device_selected': str(device), 'seed': args.seed,
                    'split_seed': cfg['split_seed'], 'synthetic': args.synthetic,
                    'source': source_fingerprint(),
                    'session_started_utc': datetime.now(timezone.utc).isoformat()})
        if not (run_dir / 'environment.json').exists():
            write_json(run_dir / 'environment.json', env)
        with (run_dir / 'sessions.jsonl').open('a') as f:
            f.write(json.dumps(env) + '\n')
        complete = train(model, loaders, device, cfg, args, run_dir)
        if not complete:
            print('Chunk complete. Resume with --resume; test evaluation waits until all epochs finish.')
            return
        if args.train_only:
            print('Training complete. Freeze choices using validation, then run --eval-only on best.pt.')
            return
    checkpoint = Path(args.checkpoint) if args.eval_only else run_dir / 'best.pt'
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    if state['config'] != cfg or state['mode'] != args.mode or state['seed'] != args.seed or state['synthetic'] != args.synthetic:
        raise ValueError('Evaluation checkpoint does not match config/mode/seed/dataset.')
    model.load_state_dict(state['model'])
    evaluate(model, loaders['test'], device, cfg, args, run_dir, checkpoint)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile mnist_robust/model.py
"""Madry MNIST ağının ölçeklenebilir iki-convolution sürümü."""

import torch
from torch import nn


class SmallCNN(nn.Module):
    """Logits üretir; girişler normalize edilmemiş [0, 1] MNIST pikselleridir.

    width=32, hidden=1024 yazarların MNIST ağındaki katman boyutlarıdır.
    Daha küçük varsayılan ağ hızlı ilk deneyler için seçilmiştir.
    """

    def __init__(self, width: int = 16, hidden: int = 128):
        super().__init__()
        for name, value in (("width", width), ("hidden", hidden)):
            if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
                raise ValueError(f"{name} pozitif bir tamsayı olmalı.")
        self.width = width
        self.hidden = hidden
        self.features = nn.Sequential(
            nn.Conv2d(1, width, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(width, 2 * width, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(7 * 7 * 2 * width, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


In [ ]:
%%writefile mnist_robust/summarize.py
"""Aggregate identical protocols across independent training seeds (sample SD)."""
import argparse
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

METRICS = ['clean_accuracy', 'fgsm_accuracy', 'pgd_accuracy', 'combined_accuracy']
KEYS = ['dataset', 'config_name', 'protocol_id', 'mode', 'n_test', 'eval_steps', 'eval_restarts', 'epsilon']


def summarize(runs, output, allow_single_seed=False):
    paths = sorted(Path(runs).glob('**/evaluations/*/metrics.csv'))
    if not paths:
        raise FileNotFoundError(f'No evaluations found in {runs}; run experiment first.')
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame['source'] = str(path)
        frames.append(frame)
    data = pd.concat(frames, ignore_index=True)
    required = KEYS + ['seed'] + METRICS
    if set(required) - set(data.columns) or data[required].isna().any().any():
        raise ValueError('Evaluation rows have missing required fields.')
    if not np.isfinite(data[METRICS].to_numpy(dtype=float)).all() or not data[METRICS].ge(0).all().all() or not data[METRICS].le(1).all().all():
        raise ValueError('Accuracy measurements must be finite fractions in [0, 1].')
    if ((data.fgsm_accuracy > data.clean_accuracy) | (data.pgd_accuracy > data.clean_accuracy)
            | (data.combined_accuracy > data[['fgsm_accuracy', 'pgd_accuracy']].min(axis=1))).any():
        raise ValueError('Robust/combined accuracy violates clean-correctness inclusion.')
    duplicate_keys = KEYS + ['seed']
    repeated = data[data.duplicated(duplicate_keys, keep=False)]
    for _, group in repeated.groupby(duplicate_keys, dropna=False):
        identical = METRICS + ['checkpoint_sha256', 'selected_epoch']
        if (set(identical) - set(group.columns) or group[identical].isna().any().any()
                or any(group[column].nunique() != 1 for column in identical)):
            raise ValueError('Duplicate seed for an identical protocol has conflicting or unverifiable evaluations. Use separate run roots.')
    if len(repeated):
        print('Identical repeat evaluations of the same checkpoint are counted once per training seed.')
        data = data.drop_duplicates(duplicate_keys, keep='first')
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    data.to_csv(output / 'per_seed.csv', index=False)
    rows = []
    for key, group in data.groupby(KEYS, dropna=False, sort=True):
        row = dict(zip(KEYS, key))
        row['n_seeds'] = len(group)
        row['seeds'] = ','.join(map(str, sorted(group.seed.tolist())))
        row['status'] = ('SMOKE_ONLY' if row['config_name'] == 'smoke' or row['dataset'] == 'synthetic'
                         else ('INCOMPLETE_SINGLE_SEED' if len(group) < 2 else 'MEASURED'))
        for metric in METRICS:
            row[metric + '_mean'] = group[metric].mean()
            row[metric + '_std'] = group[metric].std(ddof=1)
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(output / 'summary.csv', index=False)
    lines = ['# Measured outputs', '', 'Accuracies are percentages below; CSV stores fractions. '
             '± is sample standard deviation across training seeds (ddof=1), not a confidence interval. '
             'One seed has undefined SD. SMOKE_ONLY is a pipeline check, never evidence of reproduction.', '',
             '| Dataset/config/protocol | Mode | Seeds | Clean | FGSM | PGD | Combined | Status |',
             '|---|---|---|---|---|---|---|---|']
    for row in rows:
        def fmt(metric):
            mean, sd = row[metric+'_mean'], row[metric+'_std']
            return f'{100*mean:.2f} ± {100*sd:.2f}' if pd.notna(sd) else f'{100*mean:.2f} (SD undefined)'
        values = [fmt(metric) for metric in METRICS]
        lines.append(f"| {row['dataset']}/{row['config_name']}/{row['protocol_id']} | {row['mode']} | {row['seeds']} | " + ' | '.join(values) + f" | {row['status']} |")
    lines += ['', 'Combined accuracy survives clean + FGSM + every retained PGD candidate. '
              'PGD accuracy is an empirical result for the stated attack; it does not certify robustness.',
              'Error bars show descriptive sample SD, not bounded confidence intervals; mean ± SD may extend outside 0–100%.']
    (output / 'summary.md').write_text('\n'.join(lines) + '\n')
    groups = list(summary.groupby(['dataset', 'config_name', 'protocol_id'], sort=True))
    fig, axes = plt.subplots(len(groups), 1, figsize=(9, 4.2*len(groups)), squeeze=False)
    for ax, ((dataset, config, protocol), group) in zip(axes[:, 0], groups):
        x = np.arange(len(METRICS))
        width = 0.75 / len(group)
        lower, upper = 0.0, 100.0
        for i, (_, row) in enumerate(group.iterrows()):
            means = [100*row[m+'_mean'] for m in METRICS]
            sd = np.array([100*row[m+'_std'] for m in METRICS])
            finite = np.isfinite(sd)
            if finite.any():
                mean_array = np.asarray(means)
                lower = min(lower, float(np.min(mean_array[finite] - sd[finite])))
                upper = max(upper, float(np.max(mean_array[finite] + sd[finite])))
            # Do not draw a zero-width error bar for an undefined single-seed SD.
            ax.bar(x + (i - (len(group)-1)/2)*width, means, width,
                   yerr=sd if np.isfinite(sd).all() else None, capsize=4,
                   label=f"{row['mode']} (n={row['n_seeds']})")
        ax.set_xticks(x, ['Clean', 'FGSM', 'PGD', 'Combined'])
        pad = 0.02 * (upper - lower)
        ax.set_ylim(lower - pad if lower < 0 else 0, upper + pad if upper > 100 else 100)
        ax.set_ylabel('Accuracy (%)')
        suffix = ' — PIPELINE CHECK ONLY' if config == 'smoke' or dataset == 'synthetic' else ''
        ax.set_title(f'{dataset} / {config}: PGD-{int(group.iloc[0].eval_steps)} × {int(group.iloc[0].eval_restarts)}{suffix}')
        ax.legend()
    fig.tight_layout()
    fig.savefig(output / 'accuracy.png', dpi=180)
    plt.close(fig)
    histories = sorted(Path(runs).rglob('history.csv'))
    if histories:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for path in histories:
            h = pd.read_csv(path)
            name = '/'.join(path.parts[-4:-1])
            axes[0].plot(h.epoch, h.val_clean_accuracy*100, label=name)
            axes[1].plot(h.epoch, h.val_pgd_accuracy*100, label=name)
        for ax, title in zip(axes, ['Validation clean', 'Validation PGD (configuration-specific attack)']):
            ax.set(title=title, xlabel='Epoch', ylabel='Accuracy (%)', ylim=(0, 100))
            ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(output / 'training_curves.png', dpi=180)
        plt.close(fig)
    print('\n'.join(lines))
    if not allow_single_seed and (summary.n_seeds < 2).any():
        raise SystemExit('Outputs saved; at least one group has one seed. Add seeds, or use --allow-single-seed for smoke/pilot inspection.')
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--runs', default='runs')
    parser.add_argument('--output', default='results')
    parser.add_argument('--allow-single-seed', action='store_true')
    args = parser.parse_args()
    summarize(args.runs, args.output, args.allow_single_seed)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile mnist_robust/utils.py
"""Seed ve ortam kaydı: sonuçlar kullanılan koşullarla birlikte saklanır."""

from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
import os
from pathlib import Path
import platform
import random
import sys

import numpy as np
import torch


def source_fingerprint():
    """Hash the executed package sources, including standalone notebook exports.

    This records future runs only; it cannot recover historical source versions.
    """
    files = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
             for path in sorted(Path(__file__).parent.glob('*.py'))}
    digest = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()
    return {'sha256': digest, 'files': files}


def seed_everything(seed: int):
    """Aynı ortamda tekrar çalıştırılabilirlik; farklı donanımlar eşitlik garantisi vermez."""
    if isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32:
        raise ValueError("seed, [0, 2**32) aralığında bir tamsayı olmalı.")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Bazı cihazlarda deterministik implementasyonu olmayan operatörleri bildir.
    torch.use_deterministic_algorithms(True, warn_only=True)


def resolve_device(requested="auto"):
    """İstenen cihaz yoksa sessizce fallback yapmak yerine anlaşılır hata verir."""
    if requested not in {"auto", "cpu", "cuda", "mps"}:
        raise ValueError("device auto, cpu, cuda veya mps olmalı.")
    has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "mps" if has_mps else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA bulunamadı. Colab/Kaggle GPU'yu etkinleştirin veya --device cpu kullanın.")
    if requested == "mps" and not has_mps:
        raise RuntimeError("Bu ortamda MPS kullanılamıyor; --device cpu kullanın.")
    return torch.device(requested)


def environment_info():
    """JSON'a yazılabilir temel sürüm/donanım bilgileri; torchvision import edilmez."""
    try:
        torchvision_version = version("torchvision")
    except PackageNotFoundError:
        torchvision_version = None
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": str(torch.__version__),
        "torchvision": torchvision_version,
        "numpy": np.__version__,
        "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "cuda_available": torch.cuda.is_available(),
        "cuda_devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "mps_available": bool(hasattr(torch.backends, "mps") and torch.backends.mps.is_available()),
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "deterministic_warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
    }


In [ ]:
%%writefile configs/main.json
{
  "name": "main", "epochs": 10, "batch_size": 128, "lr": 0.001,
  "width": 16, "hidden": 128, "split_seed": 2026, "val_size": 5000,
  "train_limit": null, "val_limit": null, "test_limit": null, "num_workers": 0,
  "epsilon": 0.3, "train_alpha": 0.01, "train_steps": 40,
  "val_steps": 40, "val_restarts": 1,
  "eval_alpha": 0.01, "eval_steps": 100, "eval_restarts": 10, "eval_seed": 12345
}


In [ ]:
%%writefile configs/main_curriculum_pgd10_to_pgd40.json
{
  "name": "main_curriculum_pgd10_to_pgd40",
  "epochs": 10,
  "batch_size": 128,
  "lr": 0.001,
  "width": 16,
  "hidden": 128,
  "split_seed": 2026,
  "val_size": 5000,
  "train_limit": null,
  "val_limit": null,
  "test_limit": null,
  "num_workers": 0,
  "epsilon": 0.3,
  "train_alpha": 0.01,
  "train_steps": 40,
  "val_steps": 40,
  "val_restarts": 1,
  "eval_alpha": 0.01,
  "eval_steps": 100,
  "eval_restarts": 10,
  "eval_seed": 12345,
  "curriculum": {
    "epochs_1_to_3": {
      "source_config": "main_pgd10_probe",
      "train_steps": 10
    },
    "epochs_4_to_10": {
      "train_steps": 40
    }
  }
}

In [ ]:
%%writefile configs/main_pgd10_probe.json
{
  "name": "main_pgd10_probe",
  "epochs": 3,
  "batch_size": 128,
  "lr": 0.001,
  "width": 16,
  "hidden": 128,
  "split_seed": 2026,
  "val_size": 5000,
  "train_limit": null,
  "val_limit": null,
  "test_limit": null,
  "num_workers": 0,
  "epsilon": 0.3,
  "train_alpha": 0.01,
  "train_steps": 10,
  "val_steps": 40,
  "val_restarts": 1,
  "eval_alpha": 0.01,
  "eval_steps": 100,
  "eval_restarts": 10,
  "eval_seed": 12345
}

In [ ]:
%%writefile configs/pilot.json
{
  "name": "pilot", "epochs": 3, "batch_size": 128, "lr": 0.001,
  "width": 16, "hidden": 128, "split_seed": 2026, "val_size": 5000,
  "train_limit": 10000, "val_limit": 1000, "test_limit": 1000, "num_workers": 0,
  "epsilon": 0.3, "train_alpha": 0.01, "train_steps": 10,
  "val_steps": 10, "val_restarts": 1,
  "eval_alpha": 0.01, "eval_steps": 40, "eval_restarts": 3, "eval_seed": 12345
}


In [ ]:
%%writefile configs/smoke.json
{
  "name": "smoke", "epochs": 1, "batch_size": 64, "lr": 0.001,
  "width": 8, "hidden": 32, "split_seed": 2026, "val_size": 5000,
  "train_limit": 256, "val_limit": 64, "test_limit": 64, "num_workers": 0,
  "epsilon": 0.3, "train_alpha": 0.01, "train_steps": 2,
  "val_steps": 3, "val_restarts": 1,
  "eval_alpha": 0.01, "eval_steps": 5, "eval_restarts": 2, "eval_seed": 12345
}


In [ ]:
%%writefile requirements.txt
# Keep the platform's compatible CUDA torch/torchvision pair when already installed.
# Exact versions and hardware are saved into each run's environment.json.
torch>=2.5,<3
torchvision>=0.20,<1
numpy>=1.26,<3
matplotlib>=3.8,<4
pandas>=2.2,<4


In [ ]:
%%writefile README.md
# CMP4509 — Adversarial training, instability, and a PGD curriculum on MNIST

[Public repository](https://github.com/emirbakac/cmp4509-madry-mnist) · [Open standalone notebook in Colab](https://colab.research.google.com/github/emirbakac/cmp4509-madry-mnist/blob/main/notebooks/01_mnist_reproduction.ipynb) · [Original final-test CSV](results/FINAL_TEST_ALL_MODELS.csv) · [Recorded-results analysis notebook](notebooks/02_verify_recorded_results.ipynb) · [Audit and verification](docs/FINAL_AUDIT.md)

This project investigates the central methodology of [Madry et al. (ICLR 2018)](https://arxiv.org/abs/1706.06083): training against projected gradient descent (PGD) adversarial examples can improve resistance to norm-bounded attacks. A smaller PyTorch CNN and a limited training budget make this a **scaled-down methodological reproduction**, not an exact reproduction of the paper's numerical results.

We compare clean training, PGD40 adversarial training from random initialization, and a PGD10→PGD40 curriculum across seeds **0, 1, and 2**. The recorded measurements show strong attack vulnerability after clean training, severe variability after PGD40 training, and consistent learning in the three curriculum runs. Failed runs remain in every aggregate.

## Recorded final-test results

The original [nine-row CSV](results/FINAL_TEST_ALL_MODELS.csv) records **10,000 MNIST test images per model**, selected checkpoint epochs, accuracy fractions, and checkpoint SHA-256 identifiers. The experimenters identify its protocol as **L∞ ε=0.3, α=0.01, PGD100 with five random restarts**. The CSV does not itself store the attack settings or per-example predictions; those historical details require the original run logs for independent replay verification.

The following values are recomputed directly from that CSV. Units are percent; uncertainty is **sample standard deviation across three training seeds, ddof=1**, not standard error or a confidence interval.

| Training method | Clean | FGSM | PGD100×5 |
|---|---:|---:|---:|
| Standard baseline | 99.03 ± 0.05 | 11.49 ± 1.25 | 0.00 ± 0.00 |
| PGD40 from scratch | 38.12 ± 46.37 | 34.41 ± 39.95 | 31.28 ± 34.53 |
| PGD10→PGD40 curriculum | 97.85 ± 0.30 | 93.91 ± 0.62 | 88.13 ± 0.87 |

PGD40 seeds 1 and 2 each record **11.35% clean, FGSM, and PGD accuracy**, with selected epoch 1; seed 0 records 71.15% PGD accuracy at selected epoch 10. The experimenters report near-single-class prediction collapse and completion of ten training epochs. The CSV independently establishes these scores and selected epochs, but does not establish prediction histograms or the full training trajectory. Such low clean performance is not a successful robustness result. The cause of collapse, including any proposed dying-ReLU explanation, remains unproven.

The curriculum was introduced after observing PGD40 instability. The result supports this intervention under the studied settings; three seeds do not establish a universal ranking of training methods. No recorded measurement was changed during the audit. See [results provenance](results/README.md), [protocol metadata](results/FINAL_TEST_PROTOCOL.json), and [recomputed summaries](results/verified/SUMMARY.md).

## Separate stronger validation checks

The experimenters additionally report evaluations on the same **512 validation images**, using curriculum checkpoints:

| Reported validation attack | Seed 0 | Seed 1 | Seed 2 | Reported mean ± sample SD (%) |
|---|---:|---:|---:|---:|
| PGD100×20 | 84.38 | 86.52 | 83.98 | 84.96 ± 1.37 |
| Full standard AutoAttack | 82.62 | 85.16 | 82.23 | 83.34 ± 1.59 |

These rounded values are **author-reported secondary evidence**: raw attack logs, subset indices, exact counts, and original checkpoint files were not available for independent replay during this audit. They must not be relabeled as independently verified measurements. The reported AutoAttack components are APGD-CE, targeted APGD, targeted FAB, and Square; an increased Square budget reportedly reduced seed 0 to approximately 82.42%. Preserve the reported warning that Square found additional adversarial examples: **gradient masking has not been conclusively ruled out**. [AutoAttack reference and official code](https://github.com/fra31/auto-attack)

The 512-image validation checks and 10,000-image final test use different examples and budgets. Their scores are not interchangeable. None of these attack measurements is a mathematical robustness certificate. The repository's standard experiment command runs FGSM/PGD; it does not reproduce the historical AutoAttack session.

## Experimental setup

| Component | Configuration |
|---|---|
| Data | MNIST: fixed 55,000 train / 5,000 validation / 10,000 test; split seed 2026 |
| Inputs | `ToTensor()`, pixels in `[0,1]`, no mean/std normalization |
| Network | 5×5 convolutions 1→16→32, ReLU and 2×2 pooling after each; 1568→128→10 classifier |
| Trainable parameters | **215,370** |
| Optimizer | Adam, learning rate `0.001`, batch 128, ten epochs |
| Training attack | ε=0.3, α=0.01, one randomized PGD start; 40 steps or the curriculum below |
| Routine validation | PGD40×1, attack seed 12345, fixed validation split |
| Checkpoint selection | Baseline: highest clean validation accuracy; adversarial: highest PGD validation accuracy; exact ties keep the earlier checkpoint |
| Frozen final test | Clean, FGSM, PGD100×5 on selected `best.pt`; all 10,000 test images |

The code separates training/validation from test evaluation when `--train-only` is used. It does not use test scores to select checkpoints. The available CSV alone cannot reconstruct every historical model-selection decision.

**Curriculum provenance matters.** The reported historical runs first trained separate `main_pgd10_probe` models for three epochs, then transferred model, Adam, RNG, and epoch state into curriculum run directories and continued under PGD40. The current integrated `main_curriculum_pgd10_to_pgd40` configuration directly schedules **epochs 1–3: PGD10; epochs 4–10: PGD40**. It reproduces the intended schedule without requiring a separate probe run. The original transfer procedure and integrated execution are distinct provenance paths; checkpoint identity or bitwise equivalence has not been established. Do not copy a probe checkpoint into a new config directory as an undocumented resume workaround.

The [authors' MNIST implementation](https://github.com/MadryLab/mnist_challenge) uses 32/64 convolution channels, 1024 hidden units, Adam at `1e-4`, batch 50, and 100,000 updates. Our model capacity, optimization budget, initialization, validation selection, and final attack strength differ. The curriculum is our experimental intervention. The paper's PGD100×20 evaluation and our PGD100×5 test are not an equal-budget numerical comparison.

## Reproduce the workflow

### Standalone Colab/Kaggle notebook

Open [the notebook](notebooks/01_mnist_reproduction.ipynb), select a GPU, and run top to bottom. It writes its embedded source/config files and needs no repository clone. First use the default **`smoke`** (256 train / 64 validation / 64 test examples, one epoch). This verifies software operation, not scientific performance.

For the final three strategies, use **separate notebook runs**:

| `CONFIG_NAME` | Modes selected by default | Seeds | Purpose |
|---|---|---|---|
| `smoke` / `pilot` | baseline, adversarial | 0 | Software check / exploratory budget |
| `main` | baseline, adversarial | 0, 1, 2 | Clean training and PGD40 from scratch |
| `main_pgd10_probe` | adversarial | 0, 1, 2 | Optional three-epoch probe; not required by integrated curriculum |
| `main_curriculum_pgd10_to_pgd40` | adversarial | 0, 1, 2 | Direct ten-epoch 3+7 curriculum |

Use `USE_GOOGLE_DRIVE=True` in Colab for persistent checkpoints; Kaggle uses `/kaggle/working` and requires **Save Version** to retain outputs. `RESUME=True` resumes existing `last.pt` files. `EPOCHS_PER_CALL=1` limits each invocation to one additional epoch; a partially completed epoch is rerun after interruption. Existing completed training is not restarted. Save the environment/config screen and result figures, as well as the underlying CSV files.

Keep `PROTOCOL_FROZEN=False` while making validation-based decisions. For the final `main` or curriculum test, set **both `PROTOCOL_FROZEN=True` and `FINAL_STRONG_EVAL=True`**. The notebook then explicitly passes **100 steps and five restarts**. Config defaults retain the historical ten-restart setting to preserve compatibility with saved checkpoints; the explicit final override is required. Twenty restarts refer to the separate validation study.

### Command line

In a fresh Python environment, install `requirements.txt`. In Colab/Kaggle, preserve the platform's compatible CUDA `torch`/`torchvision` pair. The notebook installs only missing dependencies.

```bash
python -m pip install -r requirements.txt
python -m unittest discover -s tests -v

# First smoke check; add --synthetic only for an offline software test.
python -m mnist_robust.experiment --config configs/smoke.json --mode baseline --seed 0 --output-dir runs --device auto

# Repeat each of these three commands for seeds 0, 1, and 2.
python -m mnist_robust.experiment --config configs/main.json --mode baseline --seed 0 --output-dir runs --device auto --train-only
python -m mnist_robust.experiment --config configs/main.json --mode adversarial --seed 0 --output-dir runs --device auto --train-only
python -m mnist_robust.experiment --config configs/main_curriculum_pgd10_to_pgd40.json --mode adversarial --seed 0 --output-dir runs --device auto --train-only
```

Resume with the same config and `--resume`; optionally add `--epochs-per-call 1`. Config changes require a separate run location. Full training and strong evaluation require substantial GPU time and are not launched by the audit tests.

After freezing decisions, evaluate every selected checkpoint with the same final protocol. For example:

```bash
python -m mnist_robust.experiment \
  --config configs/main_curriculum_pgd10_to_pgd40.json \
  --mode adversarial --seed 0 --output-dir runs --device auto --eval-only \
  --checkpoint runs/main_curriculum_pgd10_to_pgd40/adversarial/seed_0/best.pt \
  --eval-steps 100 --eval-restarts 5

python -m mnist_robust.summarize --runs runs/main --output results/main
python -m mnist_robust.summarize --runs runs/main_curriculum_pgd10_to_pgd40 --output results/curriculum
```

`last.pt` stores resume state; `best.pt` stores the validation-selected model. New evaluations use unique directories under `evaluations/`, retaining previous results. `latest_evaluation.json` identifies the latest files. Identical repeated measurements from the same checkpoint are counted once by the summarizer; conflicting duplicate seeds are rejected. Distinct configs, datasets, source/protocol fingerprints, and attack budgets are not pooled.

### Notebook generation and checks

The separate [recorded-results notebook](notebooks/02_verify_recorded_results.ipynb) independently recomputes the archived statistics and figures without model training. It verifies arithmetic and supplied data provenance, not the historical attack execution. The equivalent local command is `python tools/verify_results.py` (`--no-figures` for numerical checks only).

```bash
python tools/build_notebook.py
python tools/build_notebook.py --check
python tools/build_results_notebook.py
python tools/build_results_notebook.py --check
python -m unittest discover -s tests -v
```

The generator embeds current package files, all configs, documentation, and archived result tables. Regenerate it after changing those inputs. Generation is deterministic; runtime checkpoints and archives naturally contain execution-specific data. [Validation record](docs/VALIDATION.md) distinguishes executed checks, static inspection, historical claims, and work not replayed.

## Files and submission

- `mnist_robust/`: model, data, attacks, training/evaluation, seed controls, aggregation.
- `configs/`: smoke, pilot, main, PGD10 probe, and integrated curriculum.
- `notebooks/` and `tools/build_notebook.py`: standalone notebook and its source generator.
- `results/`: original final measurements, derived summaries/figures, and clearly labeled software smoke outputs.
- `docs/`: [theory](docs/THEORY.md), [experiment/provenance guide](docs/EXPERIMENT_PLAN.md), [verification](docs/VALIDATION.md), [course deliverables](docs/COURSE_DELIVERABLES.md), and [bibliography](docs/references.bib).

Large datasets/checkpoints and personal storage paths are not committed. Recorded checkpoint hashes can only be independently verified against the original files when those files are available. The course report must use the supplied LaTeX template and be submitted as PDF by **13 December 2026, 23:59 Türkiye time**. The supplied report sources are in [report/](report/README.md). The original course template archive has not yet been located, so template provenance remains a submission blocker; resemblance to the course layout is insufficient. The repository's audit records compilation, PDF inspection and remaining requirements. Slides are a separate stage.


In [ ]:
%%writefile docs/EXPERIMENT_PLAN.md
# Deney protokolü ve yeniden üretim rehberi

Proje artık bir başlangıç taslağı değil. `results/FINAL_TEST_ALL_MODELS.csv`, üç yöntem × üç seed için dokuz kayıt içeriyor. Bu belge mevcut sonuçların kökenini yeni yeniden üretim koşularından ayırır.

## Korunacak deney ayrımları

| Deney | Veri | Bütçe / amaç | Kanıt durumu |
|---|---|---|---|
| Ana final test | 10.000 MNIST test görüntüsü | PGD100×5, ε=.3, α=.01; clean/FGSM de ölçüldü | Dokuz satırlık özgün CSV mevcut; ortalamalar yeniden hesaplanabilir |
| Güçlü validation | Aynı olduğu bildirilen 512 validation görüntüsü | PGD100×20 ve full standard AutoAttack | Yazarların bildirdiği özet değerler; ham log/indeksler olmadan yeniden doğrulanmış sayılmaz |
| Rutin checkpoint seçimi | Sabit 5.000 validation görüntüsü | Her epoch PGD40×1; baseline için temiz doğruluk | Güncel kaynak kodunda incelenebilir |
| Smoke | 256/64/64 eğitim/validation/test | Küçük yazılım doğrulaması | Ana bilimsel sonuçlarla birleştirilmez |

Ana CSV'de saldırı bütçesi sütunları ve örnek bazında tahminler yoktur. PGD100×5 protokol tanımı tarihî deney açıklamasına dayanır; CSV kontrolü bu saldırı çalışmasının bağımsız tekrarı değildir. Kaydedilmiş SHA-256 değerlerini doğrulamak için özgün checkpoint dosyaları gerekir.

## Üç yöntem ve tarihî curriculum

1. **Baseline:** 10 epoch temiz örneklerle eğitim.
2. **PGD40 from scratch:** 10 epoch, ilk batch'ten itibaren PGD40. Başarısız seed1/2 çıkarılmaz: CSV'de clean/FGSM/PGD=.1135, seçilen epoch1. Tek sınıfa çökme ve 10 epoch'un tamamlanması yazarların süreç açıklamasıdır; CSV tek başına tahmin dağılımını veya bütün geçmişi kanıtlamaz. Ölü ReLU gibi nedensel açıklamalar hipotez olarak kalır.
3. **Curriculum:** epoch1–3 PGD10, epoch4–10 PGD40. Tarihî koşularda önce üç epoch'luk probe checkpoint'leri ayrı curriculum klasörlerine taşınarak model, Adam, RNG ve epoch durumunun korunduğu bildiriliyor. Güncel entegre program aynı 3+7 saldırı bütçesini sıfırdan uygular; iki yürütme yolunun bit düzeyinde eşdeğer olduğu gösterilmedi.

`main_pgd10_probe` araştırma tarihini belgeleyen ek config'tir. Entegre curriculum koşusu için önce probe eğitmek **gerekmez**. Sadece `--resume` kullanarak farklı config'li probe checkpoint'ini yeni koşuya aktarmak desteklenen bir yöntem değildir; uyumsuzluk denetimini kaldırma.

## Yeniden çalıştırma sırası

1. Notebook'u temiz oturumda `smoke` ile çalıştır. Cihaz, config ve tamamlanma ekranını sakla.
2. Gerekirse `pilot` ile süreyi ölç. Test için değil, kaynak bütçesi ve doğrulama davranışı için kullan.
3. `main` için baseline ve adversarial; `main_curriculum_pgd10_to_pgd40` için adversarial; her birinde seed0/1/2. Bunlar ayrı notebook config koşularıdır.
4. Eğitimde `--train-only` veya notebook'ta `PROTOCOL_FROZEN=False`. Checkpoint seçimi yalnız validation ile yapılır. Baseline temiz doğruluğu, adversarial modeller PGD doğruluğunu en yüksek yapan checkpoint'i seçer. Tam eşitlikte önceki checkpoint korunur.
5. Ayarlar dondurulduktan sonra bütün checkpoint'lerde açıkça `--eval-steps 100 --eval-restarts 5` kullan. Notebook'ta `PROTOCOL_FROZEN=True` ve `FINAL_STRONG_EVAL=True` seç. Tarihî config'lerin varsayılan10 restart değerini final protokol sanma.
6. Üç seed'in bütün değerlerini, ortalama ve **örnek std (ddof=1)** ile raporla. Farklı protokolleri veya kaynak sürümlerini tek ortalamada toplama.

Free GPU oturumlarında Drive kaydı ve `EPOCHS_PER_CALL=1` kullanılabilir. Son tamamlanan epoch'tan devam edilir; kesilen epoch baştan yürür. Her epoch'un tam state'i `last.pt`, seçilen state `best.pt` olarak tutulur. Yeni değerlendirmeler önceki dosyaları değiştirmeden ayrı klasöre yazılır; `latest_evaluation.json` son konumu gösterir.

## Sonuçların yorumu

CSV'ye göre curriculum PGD100×5 doğruluğu **88.13±0.87%**; baseline **0.00±0.00%**; PGD40 **31.28±34.53%**. Son sayının yüksek yayılımı başarısız seed'leri açık eder. Normal doğruluk ve eğitim kararlılığı birlikte değerlendirilmelidir. Üç seed, yöntemin bütün ağlarda daha üstün olduğunu kanıtlamaz.

512 örnek validation skorlarını 10.000 test skoru ile eşdeğer protokol gibi kıyaslama. AutoAttack'in Square uyarısı ve ek bütçeyle bulunan hatalar saklanmamalı; gradient masking kesin olarak dışlanmış değildir. Matematiksel sertifika iddiası kurulmaz.

Tam eğitim geçmişleri mevcut değilse öğrenme eğrisi uydurma. Ayrı seed'lerin sonuçlarını epoch eğrisi gibi çizme. CSV'deki oranlar `[0,1]` aralığındadır; yüzde grafiğinde 100 ile çarpılır. Numbers'ta ondalık ayracı yanlış görünürse ham CSV'yi değiştirme.

## Deney kaydı

Yeni koşularda kaynak commit/fingerprint, config, seed, cihaz/ortam, checkpoint SHA-256, seçilen epoch, saldırı bütçesi, örnek sayısı ve çıktı konumu birlikte saklanmalı. Ekran görüntüleri anlatımı destekler; sayısal kayıtların yerine geçmez. Eski ölçüm dosyalarının üzerine yeni sonuç yazma.


In [ ]:
%%writefile docs/COURSE_DELIVERABLES.md
# CMP4509 teslim durumu ve son kontrol listesi

Tarihler Türkiye saatine göredir. Paper sign-up tamamlandı; proje dokuz modelin final-test ölçümünü içeriyor. Eğitim sonuçlarının artık var olması, bütün teslim koşullarının tamamlandığı anlamına gelmez.

| Teslim | Şart | Kanıt / kalan adım |
|---|---|---|
| Public GitHub | Kaynak, notebook, yeniden üretim adımları ve raporda bağlantı | [Public depo](https://github.com/emirbakac/cmp4509-madry-mnist) mevcut; audit branch/PR durumu FINAL_AUDIT'te |
| Bağımsız eğitim notebook'u | Kaynak/config dosyalarını içermeli; smoke ve final100×5 akışı | `notebooks/01_mnist_reproduction.ipynb`; güncel çalıştırma kanıtı VALIDATION/FINAL_AUDIT'te |
| Sonuç analizi | Özgün dokuz kayıt korunmalı; ortalama ± örnek std | `results/FINAL_TEST_ALL_MODELS.csv`, provenance JSON ve yeniden hesaplanan çıktılar |
| Rapor PDF | **13 Aralık 2026, 23:59**, Teams; gerçek ders LaTeX şablonu | `report/` içinde teslim edilen kaynaklar mevcut; derleme/inceleme sonucu FINAL_AUDIT'te |
| Resmî şablon kökeni | Teams'teki `CMP4509_report_template.zip` ile doğrulama | Özgün şablon arşivi bulunmadığı sürece **teslim engeli**; mevcut kaynakların benzer görünmesi kanıt değil |
| Sunum | 30 Kasım veya 7 Aralık 2026; QA dahil 10 dakika | Ayrı aşama; bu final audit'te slayt üretilmiyor |

## Teslimden önce

1. Özgün ders şablonunu temin et ve rapor biçimiyle karşılaştır. Alternatif bir tasarımı sessizce ders şablonu diye kullanma.
2. Son raporun, kontrol edilen `report/main.tex` ve kaynakçadan üretildiğini doğrula. Kılavuz kutuları kapalı, denklemler/citations çözülmüş, grafikler okunaklı olmalı.
3. Final tabloların **10.000 test görüntüsü, PGD100×5**; ek güçlü kontrollerin **512 validation görüntüsü, PGD100×20/AutoAttack** olduğunu ayrı belirt.
4. PGD40 seed1/2 başarısızlıklarını ve curriculum'un sonradan geliştirilen müdahale olduğunu koru. Tarihî checkpoint transferi ile entegre sıfırdan çalıştırmayı eşdeğer dosyalar gibi sunma.
5. Grup üyeleri Contributions ve Use of AI Tools metinlerini doğrulasın. Atanan sorumluluklar, gerçekten tamamlandığı doğrulanan işler gibi yazılmamalı.
6. GitHub'da gerekli kaynak/config/notebook/sonuç/rapor dosyalarının bulunduğunu kontrol et; audit değişiklikleri gözden geçirilmeden otomatik merge yapılmaz.
7. PDF'yi Teams'e yükle ve teslim onayını sakla. Bu belge otomatik hatırlatıcı veya teslim işlemi değildir.

## Saklanması yararlı kanıtlar

- Cihaz, ortam ve config ekran görüntüsü.
- Smoke notebook'un gerçek tamamlanma çıktısı.
- Seed başına özgün CSV, saldırı bütçesi, seçilen epoch, checkpoint hash'i.
- Varsa tam eğitim geçmişleri ve örnek tahminler; yoksa öğrenme eğrisi üretme.
- Ortalama ± sample SD figürleri ve doğrulanmış veri kaynağı.
- Validation saldırılarının ham logları, örnek indeksleri, AutoAttack sürümü ve Square uyarısı.
- Public repo, son PDF ve Teams teslim onayı.

CSV/config/log dosyaları temel kayıttır. Ekran görüntüleri destekleyicidir. Büyük checkpoint'ler ve MNIST verisini Git'e koymak yerine erişilebilir ayrı bir arşivde sakla; erişim anahtarlarını veya kişisel dosyaları yayımlama.


In [ ]:
%%writefile docs/THEORY.md
# Kodu anlayarak çalıştırmak: min–max, FGSM ve PGD

Elimizde bir rakam görüntüsü `x`, doğru etiket `y` ve ağırlıkları `θ` olan model `fθ` var. Normal eğitim, temiz görüntülerin cross-entropy kaybını azaltır. Adversarial eğitim, modelin küçük ama kasıtlı değişikliklere karşı da doğru kalmasını hedefler.

## 1. Min–max hedefini okuyalım

Logit vektörü `z=fθ(x)` için doğru etiketin cross-entropy kaybı:

\[
\mathcal L(z,y)=-\log\frac{\exp z_y}{\sum_{k=1}^{10}\exp z_k}.
\]

Kod `cross_entropy` fonksiyonuna doğrudan logit verir; öncesinde softmax uygulamaz. Sonlu eğitim setinde normal ampirik risk `N^{-1}∑ᵢ L(fθ(xᵢ),yᵢ)` iken dayanıklı ampirik risk her terimi `max_{δ∈S(xᵢ)} L(fθ(xᵢ+δ),yᵢ)` ile değiştirir. Aşağıdaki beklenti bunun veri dağılımı üzerindeki hedefidir.

\[
\min_\theta\; \mathbb{E}_{(x,y)}\left[
\max_{\delta\in S(x)}\mathcal{L}(f_\theta(x+\delta),y)
\right],
\qquad
S(x)=\{\delta:\|\delta\|_\infty\le\epsilon,\;x+\delta\in[0,1]^d\}.
\]

**İçteki max:** model ağırlıkları sabitken görüntüyü değiştirip kaybı büyüt. **Dıştaki min:** bu zor görüntüyü kullanıp ağırlıkları değiştirerek kaybı azalt. Veri beklentisini minibatch'lerle, içteki aramayı PGD ile yaklaşık çözüyoruz. Madry ve arkadaşlarının temel yaklaşımı dayanıklı optimizasyon çerçevesidir. [Makale](https://arxiv.org/abs/1706.06083)

L∞ uzaklığı, pikseller arasındaki en büyük mutlak farktır: `max(abs(x_adv - x))`. Bizde pikseller `[0,1]`, ε ise `0.3`. Bu sınır toplam değişimi değil **her pikselin** değişimini sınırlar. Çözünürlük, veri ölçeği veya tehdit normu değişirse aynı epsilon başka anlama gelebilir.

## 2. FGSM neden gradyanın işaretini kullanıyor?

Bir görüntünün yakınında kaybı birinci dereceden Taylor yaklaşımıyla yazalım:

\[
\mathcal{L}(x+\delta)\approx\mathcal{L}(x)+g^\top\delta,
\qquad g=\nabla_x\mathcal{L}(f_\theta(x),y).
\]

Önce ek piksel kutusu sınırını dışarıda bırakırsak, L∞ topunda bu doğrusal ifadenin maksimumu:

\[
\max_{\|\delta\|_\infty\le\epsilon}g^\top\delta
=\epsilon\sum_i |g_i|=\epsilon\|g\|_1,
\qquad \delta_i^*=\epsilon\,\operatorname{sign}(g_i).
\]

Her pikseli gradyan pozitifse artırmak, negatifse azaltmak doğrusal yaklaşımın kaybını büyütür. L∞ sınırının L1 gradyan normuyla bu ilişkisi **dual norm** ilişkisidir. Görüntü aralığını da koruyarak:

\[
x_{\rm FGSM}=\operatorname{clip}_{[0,1]}
\left(x+\epsilon\operatorname{sign}(g)\right).
\]

Taylor yaklaşımı yaklaşık olduğu için bu tek adım gerçek sinir ağında en kötü görüntüyü garanti etmez. FGSM'nin birincil kaynağı [Goodfellow, Shlens ve Szegedy](https://arxiv.org/abs/1412.6572).

## 3. PGD bu fikri tekrar ediyor

Rastgele bir başlangıç oluştur:

\[
x^{(0)}=\operatorname{clip}_{[0,1]}(x+u),\qquad
u_i\sim\mathcal{U}(-\epsilon,\epsilon).
\]

Sonra her adımda gradyanı yeniden hesapla:

\[
x^{(t+1)}=\Pi_{[x-\epsilon,x+\epsilon]\cap[0,1]^d}
\left(x^{(t)}+\alpha\operatorname{sign}
\big(\nabla_{x^{(t)}}\mathcal{L}(f_\theta(x^{(t)}),y)\big)\right).
\]

`Π` burada piksel bazında kırpmadır. Merkez daima **orijinal x** kalır; merkezi her adım kaydırırsak ε sınırını aşabiliriz. Adım boyu `α`, toplam izin verilen değişim `ε` ile aynı şey değildir. Bu güncelleme ve rastgele başlangıç [yazarların PGD kodunda](https://github.com/MadryLab/mnist_challenge/blob/master/pgd_attack.py) doğrudan görülebilir.

Bir **restart**, yeni bir rastgele başlangıçla saldırıyı baştan çalıştırır. Farklı başlangıçlar farklı zor bölgeler bulabilir. Projemizde değerlendirme bütün başlangıç/ara adım/restart adaylarında bulunan yanlış sınıflandırmayı korur. Son adımın tesadüfen tekrar doğru olması önceki saldırı başarısını silmez. Bu, resmî kodun son adayı döndürme davranışına göre ek bir değerlendirme önlemidir.

## 4. Neden saldırıda `detach`, sonra ayrı eğitim gradyanı?

İki farklı türev alıyoruz:

1. Saldırıda `∇x loss`: model sabit, değişken görüntü. Yeni saldırı adımı `detach()` edilir ve bir sonraki adım için yeniden `requires_grad` açılır. Böylece bütün PGD iterasyonlarını tek dev hesaplama grafiğine bağlamayız.
2. Eğitimde `∇θ loss`: üretilen saldırılı görüntü sabit girdi kabul edilir; yeni ileri geçişte model ağırlıklarına türev alınır ve optimizer günceller.

Bu uygulamada dış adım, PGD algoritmasının bütün hesaplamalarından geriye türev almaz. `detach`, dış kaybın model ağırlıklarına türevini engellemez; yalnız saldırı üretim zincirinden ayırır. Resmî eğitim de önce adversarial örneği üretir, sonra bu örneği eğitim adımına besler. [Resmî eğitim kodu](https://github.com/MadryLab/mnist_challenge/blob/master/train.py)

İç maksimum tam çözülebilseydi, uygun düzenlilik koşullarında bu tür dış türev için Danskin yaklaşımı kullanılabilir. Burada PGD yaklaşık çözücüdür; bu teorik bağlantı kodun küresel optimum bulduğuna dair garanti değildir.

## 5. Doğrulukta payda bütün test seti

`N` test örneği için temiz doğruluk, doğru tahminlerin sayısının `N`'ye bölümüdür. PGD doğruluğunda da payda **aynı N** kalır:

\[
\widehat{A}_{\rm PGD}=\frac1N\sum_{i=1}^N
\mathbf{1}\{f(x_i)=y_i\}
\mathbf{1}\{\text{tutulan bütün PGD adayları doğru sınıflandırıldı}\}.
\]

Yalnız temizken doğru örnekleri paydaya alıp bunu genel dayanıklı doğruluk diye sunmuyoruz. Sıfır değişim de tehdit kümesinde olduğundan temizken yanlış bir örnek dayanıklı olamaz. `combined_accuracy`, temiz + FGSM + PGD adaylarının hepsini birlikte geçme oranıdır. Restart doğruluklarının ortalaması değildir.

Bu metrik **ampirik saldırı doğruluğudur**. PGD'nin hata bulması bir zayıflık gösterir; hata bulamaması izinli bütün değişimlerin güvenli olduğunu kanıtlamaz. Daha fazla adım/restart bu aramayı güçlendirir, sertifikaya dönüştürmez. Sonuçları yorumlarken `ε`, `α`, adım, restart, test örnek sayısı ve seed'i birlikte belirt.

## 6. Bu projede hangi saldırı nerede kullanılıyor?

Eğitimden güçlü bir final testi kullanmak, bütün saldırılara dayanıklılık kanıtı değildir. Bu çalışmanın dondurulmuş **10.000 test örneği protokolü PGD100×5**; yazarların ayrıca bildirdiği **512 validation örneği kontrolü PGD100×20 ve AutoAttack**'tir. Örnek kümeleri ve bütçeler farklıdır. AutoAttack'in ek Square başarısı uyarısı varsa gradient masking'in kesin dışlandığını söyleyemeyiz. [AutoAttack'in resmî uygulaması](https://github.com/fra31/auto-attack)

Curriculum ilk üç epoch'ta PGD10, kalan yedi epoch'ta PGD40 kullanır. İç max'ın yaklaşık çözümünü eğitim boyunca değiştirir; tehdit sınırı ε değişmez. Bu, orijinal makalenin birebir protokolü değil, gözlenen kararsızlıktan sonra yapılan deneysel müdahaledir. Sonuçlar bu seçim geçmişiyle yorumlanmalıdır.

Koddaki karşılıklar: `mnist_robust/attacks.py`, `mnist_robust/experiment.py`. İlk smoke çalışmasından sonra bu dosyaları yukarıdaki iki gradyan ayrımını takip ederek okumak iyi bir ikinci adımdır.


In [ ]:
%%writefile docs/references.bib
@inproceedings{madry2018towards,
  title     = {Towards Deep Learning Models Resistant to Adversarial Attacks},
  author    = {Madry, Aleksander and Makelov, Aleksandar and Schmidt, Ludwig and Tsipras, Dimitris and Vladu, Adrian},
  booktitle = {International Conference on Learning Representations},
  year      = {2018},
  eprint    = {1706.06083},
  archivePrefix = {arXiv},
  primaryClass = {stat.ML},
  url       = {https://arxiv.org/abs/1706.06083}
}

@misc{madrylab_mnist_challenge,
  author       = {{MadryLab}},
  title        = {{MNIST Adversarial Examples Challenge}: Official Implementation},
  howpublished = {\url{https://github.com/MadryLab/mnist_challenge}},
  url          = {https://github.com/MadryLab/mnist_challenge},
  note         = {Accessed 2026-10-24; use a pinned commit when reporting the final experiment}
}

@article{goodfellow2014explaining,
  title   = {Explaining and Harnessing Adversarial Examples},
  author  = {Goodfellow, Ian J. and Shlens, Jonathon and Szegedy, Christian},
  journal = {arXiv preprint arXiv:1412.6572},
  year    = {2014},
  url     = {https://arxiv.org/abs/1412.6572}
}

@inproceedings{croce2020reliable,
  title     = {Reliable Evaluation of Adversarial Robustness with an Ensemble of Diverse Parameter-free Attacks},
  author    = {Croce, Francesco and Hein, Matthias},
  booktitle = {Proceedings of the 37th International Conference on Machine Learning},
  year      = {2020},
  url       = {https://arxiv.org/abs/2003.01690}
}

@misc{croce_autoattack_code,
  author       = {Croce, Francesco and Hein, Matthias},
  title        = {{AutoAttack}: Official Implementation},
  howpublished = {\url{https://github.com/fra31/auto-attack}},
  url          = {https://github.com/fra31/auto-attack},
  note         = {Accessed 2026-10-25; historical execution version not available in archived summaries}
}


In [ ]:
%%writefile docs/VALIDATION.md
# Verification record — 25 October 2026 audit

Verification has several distinct levels. The original measurements are preserved; checking their arithmetic does not rerun the historical training or attack jobs.

## Recorded measurements

- `results/FINAL_TEST_ALL_MODELS.csv` contains nine rows, three methods and seeds 0/1/2 for each, with 10,000 test images per row. Clean/FGSM/PGD values are fractions in `[0,1]` and the method summaries use sample SD (`ddof=1`).
- `results/FINAL_TEST_PROTOCOL.json` records the unchanged CSV digest and clearly labels the PGD100×5 attack settings as historical author-supplied provenance. Those settings are not encoded in the original compact CSV.
- `tools/verify_results.py` checks the preserved bytes, seed coverage, selected epochs, denominators, accuracy ranges and hash syntax; it recomputes statistics and derived figures. `results/verified/` contains derived arithmetic, not new model measurements.
- Recorded checkpoint hash strings cannot be matched to original full-model bytes until those checkpoint files are available.
- `results/validation_reported.json` preserves the separate rounded, author-reported 512-image PGD100×20/AutoAttack values. Raw logs, subset indices and per-example predictions are unavailable. Rounded percentages were not converted into invented observed counts.

## Notebook contract tests executed

`python -m unittest discover -s tests -p test_notebook.py -v`: **7 tests passed** during this audit. They cover deterministic generation, exact embedded package/config contents, all five config selections, seeds/modes, repeated settings in one kernel, frozen final100×5 command construction, blocked test calls before training completion/freezing, and read-only stale-notebook detection. These tests generate into temporary files and mock the expensive experiment commands; they do not train full models or run full AutoAttack.

The first execution exposed a test assertion's macOS `/var` versus `/private/var` symlink representation; resolving both paths fixed the assertion. The generated settings behavior itself was unchanged by that test correction.

`python -m py_compile tools/build_notebook.py` also completed successfully. After any further edits, regenerate `notebooks/01_mnist_reproduction.ipynb`, run `python tools/build_notebook.py --check`, and rerun the full suite. Current final full-suite and notebook execution outcomes are recorded in [FINAL_AUDIT.md](FINAL_AUDIT.md), rather than inferred from this narrower check.

## Historical software verification

The prior 24 October record reports 17 passing tests and an end-to-end real-MNIST smoke execution on CPU/Apple MPS, including checkpoint resume and artifact generation. Those checks concerned the earlier code/notebook version. They do not prove that the current notebook, CUDA full training, or historical nine-model experiment was independently replayed. Historical files in `results/smoke_check/` remain labeled as software checks.

## Limits of this audit

No nine-model retraining, full 10,000-image attack replay, or full AutoAttack run is performed for code auditing. Integrated curriculum scheduling and epoch-boundary resume can be tested cheaply, but that does not establish equivalence to the original probe-checkpoint transfer process. Cross-device bitwise equivalence is not assumed. Exact current test counts, actual notebook runs, report compilation/inspection and remaining template-provenance issues belong in [the final audit](FINAL_AUDIT.md).


In [ ]:
%%writefile results/README.md
# Results and provenance

`FINAL_TEST_ALL_MODELS.csv` is the original nine-row measurement record for Baseline, PGD40, and Curriculum, each with seeds 0, 1, and 2. Preserve its original values. Accuracy columns contain fractions in `[0,1]`; each row records 10,000 test images, the selected epoch, and a checkpoint SHA-256 identifier.

The experimenters identify the frozen final protocol as MNIST test, L∞ ε=0.3, α=0.01, PGD100 with **five restarts**. Attack settings are not columns in the original CSV; raw evaluation logs/checkpoints are needed to independently verify the historical execution, not merely its arithmetic. The audit recomputes all method means and sample standard deviations (ddof=1), including FGSM, without retraining or replacing measurements.

Reported 512-image validation PGD100×20/AutoAttack summaries are a separate evidence category. In the absence of their raw logs, per-example records, and subset indices, they remain author-reported checks rather than independently replayed results. Never pool them with the final test.

`smoke_check/` contains historical software-check outputs from a tiny dataset. Its figures are not the final experiment or full training histories. Do not use them as main-study learning curves.

New training produces `history.csv` and checkpoints under its run directory. New evaluation produces unique subdirectories under `evaluations/`, with metric, per-example, environment/source, and checkpoint provenance; `latest_evaluation.json` points to the latest output. The summarizer treats repeat measurements as one seed only when checkpoint and numerical results agree; conflicts require investigation. Generated summary figures/tables are derived artifacts, not replacement raw records.

CSV locale warning: `0.975` means 97.5%, not 975. Use a parser with decimal `.` (for example `pandas.read_csv`) and never resave distorted spreadsheet displays over the original CSV. All final plotted uncertainty is sample SD across training seeds, not standard error or confidence intervals.


In [ ]:
%%writefile results/FINAL_TEST_ALL_MODELS.csv
method,seed,best_epoch,clean_accuracy,fgsm_accuracy,pgd_accuracy,test_images,checkpoint_sha256
Baseline,0,10,0.9907,0.1238,0.0,10000,b885a6817dc618fa97b69cf0dd20e296ef2c42ce1c3c6a7756209aceab88b3c2
Baseline,1,8,0.9905,0.1203,0.0,10000,7846debdc4befa415958ca27f5284f6f2bbc1c59527479042b34d63208f389d3
Baseline,2,10,0.9897,0.1007,0.0,10000,3b260a36535c5d29cfa8b805abd53ff54deedf48699eced6445de54ba7849d18
PGD40,0,10,0.9166,0.8054,0.7115,10000,28c1a734ebe9c358c61fe7c8efd3d0f1b34cb82eac02ec11ce671a91f17a8fd5
PGD40,1,1,0.1135,0.1135,0.1135,10000,82ddd100e648840e3ed3cba94af45abec3ca204f63024f23983f96c944320697
PGD40,2,1,0.1135,0.1135,0.1135,10000,15a980aceba020af7bb9487ee7b2ad74d77990d00d48aa13a641f778876fc981
Curriculum,0,10,0.975,0.9322,0.8762,10000,4f3854078f2f4b060764ec7cab607dbe62dc68c19dc524e97fd481513c373ce7
Curriculum,1,10,0.9801,0.9442,0.8914,10000,06b78642910bafad0366fc481798f56234cec9bdf8a3404c1fb1c50b2559a330
Curriculum,2,10,0.9803,0.9409,0.8763,10000,e23b5b390403499e8fb533a8f06b8c507f214f6455be9950a4c7bc4afab1c7ad


In [ ]:
%%writefile results/FINAL_TEST_PROTOCOL.json
{
  "measurement_file": "FINAL_TEST_ALL_MODELS.csv",
  "measurement_sha256": "654e9696f9a11c86e514b44ea9bd9d38fcfb81dabb281d7482b25f8e7de5d345",
  "protocol_provenance": "Frozen protocol reported by the author and supplied manuscript. The compact original CSV itself does not encode attack settings, execution environment, data indices, or code revision.",
  "dataset": "MNIST", "split": "test", "n_examples_per_checkpoint": 10000,
  "norm": "Linf", "epsilon": 0.3, "pixel_range": [0, 1],
  "pgd_steps": 100, "pgd_alpha": 0.01, "pgd_random_restarts": 5,
  "attack_objective": "untargeted cross-entropy",
  "training_seeds": [0, 1, 2],
  "reported_checkpoint_selection": "Best clean validation for baseline, best PGD validation for adversarial conditions; first epoch wins ties in audited code.",
  "checkpoint_files_available_to_audit": false,
  "historical_execution_replayed": false,
  "historical_code_revision": null,
  "historical_eval_rng_seed": null,
  "audit_status": "Nine source rows, all aggregates, denominators, ranges, and hash syntax checked; actual trained checkpoints and historical evaluation process not independently replayed."
}


In [ ]:
%%writefile results/validation_reported.json
{
  "provenance": "Values transcribed from the supplied manuscript and the author's audit request, 25 October 2026. No raw AutoAttack logs, per-example predictions, subset index manifest, or full-model checkpoint files were available to the auditor.",
  "status": "author_reported_rounded_percentages",
  "split": "validation",
  "reported_subset_size": 512,
  "same_subset_across_methods": "reported; index identity not independently verified",
  "epsilon": 0.3,
  "pgd_alpha": 0.01,
  "seeds": [0, 1, 2],
  "attacks": [
    {"name": "PGD100 x 20", "accuracy_percent": [84.38, 86.52, 83.98]},
    {"name": "Full standard AutoAttack", "accuracy_percent": [82.62, 85.16, 82.23]}
  ],
  "autoattack_components_reported": ["APGD-CE", "targeted APGD", "targeted FAB", "Square Attack"],
  "additional_square_check": {"seed": 0, "before_percent": 82.62, "after_percent": 82.42, "exact_query_and_restart_budget": null, "status": "author_reported; raw log unavailable"},
  "aggregation": "Mean and sample SD (ddof=1) are recomputed from the supplied rounded percentages. They are not inferred exact counts out of 512.",
  "warning": "Author reports a Square Attack warning. These measurements do not establish absence of gradient masking or certified robustness. Do not pool with the 10000-image final test."
}


## 4 · Önce eğit ve doğrula

Her mode/seed ayrı klasöre kaydedilir. Bu hücre **test setini değerlendirmez**; checkpoint seçimi yalnız validation sonuçlarını kullanır. Baseline temiz doğrulama, PGD eğitimli model saldırılı doğrulama üzerinden en iyi checkpoint'i seçer. `last.pt` devam etmek, `best.pt` daha sonra test etmek içindir. Tam eşit validation skorunda önceki checkpoint korunur. Her PGD adımı için bütün gradyan zinciri tutulmaz; son saldırılı örnekten ayrı dış eğitim adımı alınır.

PGD eğitimi aynı epoch sayısında normal eğitimden çok daha pahalıdır. Pilot süresini ölçmeden ana deneyin süresini varsayma. Önbelleğe alınmamış MNIST ilk seferde indirilecektir; sentetik smoke kullanılırsa bunun gerçek veri sonucu olmadığını çıktıda koruyoruz.


In [ ]:
import time

def run_command(args):
    print("\nÇalıştırılıyor:", " ".join(str(part) for part in args), flush=True)
    subprocess.run([str(part) for part in args], cwd=PROJECT_DIR, check=True)

training_wall_seconds = {}
for mode in MODES:
    for seed in SEEDS:
        run_dir = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{seed}"
        args = [sys.executable, "-m", "mnist_robust.experiment",
                "--config", f"configs/{CONFIG_NAME}.json", "--mode", mode,
                "--seed", str(seed), "--output-dir", str(RUN_ROOT),
                "--device", "auto", "--train-only",
                "--epochs-per-call", str(EPOCHS_PER_CALL)]
        if RESUME and (run_dir / "last.pt").exists():
            args.append("--resume")
        if SYNTHETIC:
            args.append("--synthetic")
        started = time.perf_counter()
        run_command(args)
        training_wall_seconds[f"{mode}/seed_{seed}"] = time.perf_counter() - started
import csv
config_epochs = __import__("json").loads(Path(f"configs/{CONFIG_NAME}.json").read_text())["epochs"]
TRAINING_COMPLETE = True
for mode in MODES:
    for seed in SEEDS:
        history_path = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{seed}" / "history.csv"
        if history_path.exists():
            with history_path.open() as handle:
                completed_epochs = max((int(row["epoch"]) for row in csv.DictReader(handle)), default=0)
        else:
            completed_epochs = 0
        TRAINING_COMPLETE &= completed_epochs >= config_epochs
        print(f"{mode}/seed_{seed}: {completed_epochs}/{config_epochs} epoch")
print("Bütün eğitimler tamam mı?", TRAINING_COMPLETE)
print("Bu notebook oturumundaki eğitim/yeniden başlatma süreleri (s):", training_wall_seconds)
print("Ekran görüntüsü: eğitim ve doğrulama çıktıları. Checkpoint'ler:", RUN_ROOT)


## 5 · Temiz, FGSM ve güçlü PGD testi

Smoke'ta küçük test otomatik açık. Pilot/main'de `PROTOCOL_FROZEN=True` gereklidir; önceki hücrelerin tamamı çalışabilir ama bu izin verilmeden test yapılmaz.

Bir örnek herhangi bir restart/PGD adımında yanlış sınıflandırıldıysa saldırı başarılı sayılır. Restart doğruluklarının ortalamasını almak fazla iyimser sonuç verir. Temizken yanlış olan bir örnek de dayanıklı sayılmaz. FGSM tek adımdır; PGD yinelemelidir. Final PGD100×5, `main` eğitim PGD40'tan güçlüdür ve pahalı olabilir.

Bu test seçilen saldırı altında ampirik dayanıklılık ölçer; bütün olası saldırılara karşı garanti vermez. Ayrı bildirilen 512 validation örneğindeki PGD100×20/AutoAttack ölçümü bu hücre tarafından yapılmaz. Yeni değerlendirmeler benzersiz klasörlere kaydedilir; eski çıktılar korunur.


In [ ]:
EVALUATION_COMPLETED = False
if not TRAINING_COMPLETE:
    print("Eğitim parçası tamamlandı. RESUME=True ile eğitim hücresini tekrar çalıştırın; test bekliyor.")
elif not RUN_TEST_EVALUATION:
    print("Test kapalı: doğrulama sonuçlarıyla karar verin. Sonra PROTOCOL_FROZEN=True yapın.")
else:
    for mode in MODES:
        for seed in SEEDS:
            checkpoint = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{seed}" / "best.pt"
            if not checkpoint.exists():
                raise FileNotFoundError(f"Önce eğitim hücresini tamamlayın: {checkpoint}")
            args = [sys.executable, "-m", "mnist_robust.experiment",
                    "--config", f"configs/{CONFIG_NAME}.json", "--mode", mode,
                    "--seed", str(seed), "--output-dir", str(RUN_ROOT), "--device", "auto",
                    "--eval-only", "--checkpoint", str(checkpoint)]
            if SYNTHETIC:
                args.append("--synthetic")
            if FINAL_STRONG_EVAL:
                args.extend(["--eval-steps", "100", "--eval-restarts", "5"])
            run_command(args)
    EVALUATION_COMPLETED = True
    print("Test tamamlandı. Tabloyu görmeden önce config/seed/PGD bütçesini tekrar kontrol edelim.")


## 6 · CSV, tablolar ve şekiller

Her seed'in değerleri ayrıca saklanır. Ana deneyde **ortalama ± örnek standart sapma** kullanılır. Tek seed'in standart sapması hesaplanabilir bir belirsizlik ölçüsü değildir. Smoke/pilot/main veya farklı saldırı protokolleri aynı ortalamaya girmez.

**Screenshot hatırlatması:** koşul adının göründüğü sonuç tablosu, doğruluk karşılaştırması ve saldırılmış örnekler. CSV'leri de sakla; görüntü dosyası veri dosyasının yerine geçmez. Henüz koşulmamış deneyin sayısını rapora yazmayacağız.


In [ ]:
from IPython.display import display, Image, Markdown

if EVALUATION_COMPLETED:
    args = [sys.executable, "-m", "mnist_robust.summarize",
            "--runs", str(RUN_ROOT / RUN_CONFIG_NAME), "--output", str(RESULTS_DIR)]
    if len(SEEDS) < 2:
        args.append("--allow-single-seed")
    run_command(args)
    report = RESULTS_DIR / "summary.md"
    if report.exists():
        display(Markdown(report.read_text(encoding="utf-8")))
    for filename in ["accuracy.png", "training_curves.png"]:
        figure = RESULTS_DIR / filename
        if figure.exists():
            display(Image(filename=str(figure)))
    for mode in MODES:
        example_run = RUN_ROOT / RUN_CONFIG_NAME / mode / f"seed_{SEEDS[0]}"
        pointer = example_run / "latest_evaluation.json"
        if not pointer.exists():
            raise FileNotFoundError(f"Yeni değerlendirme işaretçisi bulunamadı: {pointer}")
        example = example_run / __import__("json").loads(pointer.read_text())["examples"]
        if example.exists():
            display(Markdown(f"**{CONFIG_NAME} · {mode} · seed {SEEDS[0]}**"))
            display(Image(filename=str(example)))
    print("Sonuç dosyaları:", [p.name for p in sorted(RESULTS_DIR.glob("*"))])
else:
    print("Test tablosu yok. Eğitim geçmişi ve checkpoint'ler saklandı:", RUN_ROOT)
    for history in sorted((RUN_ROOT / RUN_CONFIG_NAME).glob("*/seed_*/history.csv")):
        print("\n", history.relative_to(RUN_ROOT))
        print(history.read_text(encoding="utf-8")[-3000:])


## 7 · Çalışmayı kaybetmeden sakla

Aşağıdaki arşiv kaynak kodunu, config'leri, checkpoint'leri ve sonuçları içerir. Uzun deneylerde arşiv büyüyebilir; Drive kaydı varsa orijinal klasör de kalıcıdır. Kaggle'da **Save Version** yapıp Output bölümünü sakla. Colab'da aşağıdaki isteğe bağlı indirme hücresini çalıştır.

Notebook'un kendisini de `.ipynb` olarak indirip public GitHub deposuna ekle. Veri ve büyük checkpoint arşivlerini Git'e yükleme; README'de ayrı bir paylaşım bağlantısı kullan.


In [ ]:
import json
import shutil
import zipfile
from datetime import datetime, timezone
from uuid import uuid4

archive_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid4().hex[:8]
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "config_name": CONFIG_NAME, "modes": MODES, "seeds": SEEDS,
    "protocol_frozen": PROTOCOL_FROZEN, "final_strong_eval": FINAL_STRONG_EVAL,
    "synthetic": SYNTHETIC,
    "note": "Smoke is a pipeline check, not a scientific result.",
    "session_training_seconds": training_wall_seconds,
}
(ARTIFACT_ROOT / f"notebook_manifest_{CONFIG_NAME}_{archive_id}.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
archive_path = BASE_DIR / f"madry_mnist_{CONFIG_NAME}_{archive_id}_artifacts.zip"
with zipfile.ZipFile(archive_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:
    for root, prefix in [(PROJECT_DIR, "project"), (ARTIFACT_ROOT, "artifacts")]:
        for p in sorted(root.rglob("*")):
            if not p.is_file() or "__pycache__" in p.parts or p.suffix == ".pyc":
                continue
            if "data" in p.relative_to(root).parts:  # MNIST yeniden indirilebilir
                continue
            if p.resolve() == archive_path.resolve():
                continue
            archive.write(p, f"{prefix}/{p.relative_to(root).as_posix()}")
print("Arşiv hazır:", archive_path, f"({archive_path.stat().st_size / 1024**2:.1f} MB)")
if IS_COLAB:
    print("İndirmek için sonraki hücrede DOWNLOAD_ARCHIVE=True yapın.")
elif IS_KAGGLE:
    print("Kaggle: Save Version → Output dosyalarını saklayın.")
else:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))


In [ ]:
DOWNLOAD_ARCHIVE = False
if DOWNLOAD_ARCHIVE and IS_COLAB:
    from google.colab import files
    files.download(str(archive_path))


## Sonraki adımı veriye göre seçelim

- **Smoke tamamlandıysa:** cihaz/config/tablo ekran görüntüsünü ve arşivi sakla; pilot normal eğitimle başla.
- **Pilot tamamlandıysa:** öğrenme eğrilerini ve süreyi karşılaştır; main bütçesini kararlaştır, seed listesini sabitle.
- **Üç seed'li main eğitimleri tamamlandıysa:** ayarları dondur, bütün checkpoint'leri aynı PGD100×5 ile test et.
- **Gerçek sonuçlar hazırsa:** seed tablosu, ortalama ± std, eğitim maliyeti ve makaleden farkları raporla. Gerçek ders LaTeX şablonunu kullan; public repo bağlantısını ekle.

Sunumda her grup üyesi şu zinciri açıklayabilsin: **tehdit modeli → min–max → PGD → adversarial training → güçlü değerlendirme → sonuç ve sınırlılıklar**.
